In [ ]:
# In[1]:

import pandas as pd
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Load CSVs
metric = pd.read_csv('metric.csv')
trace = pd.read_csv('trace.csv')
log = pd.read_csv('log.csv')
error_logs = pd.read_csv('error_logs.csv')

# Helper to add localized datetime column and compute basic stats
def summarize_df(df, ts_col='timestamp'):
    # Ensure timestamp numeric
    df[ts_col] = pd.to_numeric(df[ts_col], errors='coerce')
    # Localized datetime column in UTC+8
    df['_ts_dt'] = pd.to_datetime(df[ts_col], unit='s', utc=True).dt.tz_convert(tz)
    total_rows = int(df.shape[0])
    if total_rows > 0:
        min_ts = df['_ts_dt'].min()
        max_ts = df['_ts_dt'].max()
    else:
        min_ts = pd.NaT
        max_ts = pd.NaT
    cmdb_counts = df['cmdb_id'].value_counts()
    unique_cmdb_count = int(cmdb_counts.shape[0])
    # return summary pieces
    summary = pd.DataFrame([{
        'total_rows': total_rows,
        'unique_cmdb_count': unique_cmdb_count,
        'min_timestamp': min_ts,
        'max_timestamp': max_ts
    }])
    return df, summary, cmdb_counts

# Summarize metric.csv
metric, metric_summary, metric_cmdb_counts = summarize_df(metric, 'timestamp')
metric_cmdb_top50 = metric_cmdb_counts.head(50).rename_axis('cmdb_id').reset_index(name='count')
metric_kpi_top20 = metric['kpi_name'].value_counts().head(20).rename_axis('kpi_name').reset_index(name='count')

# Summarize trace.csv
trace, trace_summary, trace_cmdb_counts = summarize_df(trace, 'timestamp')
trace_cmdb_top50 = trace_cmdb_counts.head(50).rename_axis('cmdb_id').reset_index(name='count')
trace_top20 = trace['trace_name'].value_counts().head(20).rename_axis('trace_name').reset_index(name='count')

# Summarize log.csv
log, log_summary, log_cmdb_counts = summarize_df(log, 'timestamp')
log_cmdb_top50 = log_cmdb_counts.head(50).rename_axis('cmdb_id').reset_index(name='count')
log_top20 = log['log_name'].value_counts().head(20).rename_axis('log_name').reset_index(name='count')

# Summarize error_logs.csv (include sample top 10 rows)
error_logs, error_summary, error_cmdb_counts = summarize_df(error_logs, 'timestamp')
error_cmdb_top50 = error_cmdb_counts.head(50).rename_axis('cmdb_id').reset_index(name='count')
# Top messages by frequency (top 20)
if 'message' in error_logs.columns:
    error_message_top20 = error_logs['message'].value_counts().head(20).rename_axis('message').reset_index(name='count')
else:
    error_message_top20 = pd.DataFrame(columns=['message','count'])
# Sample top 10 rows with localized timestamp, selected columns
error_sample = error_logs.loc[:, ['_ts_dt','cmdb_id','message']].head(10).rename(columns={'_ts_dt':'timestamp(Asia/Shanghai)'})

# Compact outputs (limited rows where appropriate)
metric_summary, metric_cmdb_top50, metric_kpi_top20, trace_summary, trace_cmdb_top50, trace_top20, log_summary, log_cmdb_top50, log_top20, error_summary, error_cmdb_top50, error_message_top20, error_sample

```
Out[1]:
```
```python
# Compose a plain-English compact summary string using previous analysis variables
metric_tot = int(metric_summary.iloc[0]['total_rows'])
metric_unique = int(metric_summary.iloc[0]['unique_cmdb_count'])
metric_min = str(metric_summary.iloc[0]['min_timestamp'])
metric_max = str(metric_summary.iloc[0]['max_timestamp'])
metric_top_cmdb = ", ".join([f"{r.cmdb_id}({r.count})" for r in metric_cmdb_top50.head(10).itertuples(index=False)])
metric_top_kpi = ", ".join([f"{r.kpi_name}({r.count})" for r in metric_kpi_top20.head(10).itertuples(index=False)])

trace_tot = int(trace_summary.iloc[0]['total_rows'])
trace_unique = int(trace_summary.iloc[0]['unique_cmdb_count'])
trace_min = str(trace_summary.iloc[0]['min_timestamp'])
trace_max = str(trace_summary.iloc[0]['max_timestamp'])
trace_top_cmdb = ", ".join([f"{r.cmdb_id}({r.count})" for r in trace_cmdb_top50.head(10).itertuples(index=False)])
trace_top_name = ", ".join([f"{r.trace_name}({r.count})" for r in trace_top20.head(10).itertuples(index=False)])

log_tot = int(log_summary.iloc[0]['total_rows'])
log_unique = int(log_summary.iloc[0]['unique_cmdb_count'])
log_min = str(log_summary.iloc[0]['min_timestamp'])
log_max = str(log_summary.iloc[0]['max_timestamp'])
log_top_cmdb = ", ".join([f"{r.cmdb_id}({r.count})" for r in log_cmdb_top50.head(10).itertuples(index=False)])
log_top_name = ", ".join([f"{r.log_name}({r.count})" for r in log_top20.head(10).itertuples(index=False)])

error_tot = int(error_summary.iloc[0]['total_rows'])
error_unique = int(error_summary.iloc[0]['unique_cmdb_count'])
error_min = str(error_summary.iloc[0]['min_timestamp'])
error_max = str(error_summary.iloc[0]['max_timestamp'])
# error sample existence
error_sample_exists = False if error_tot == 0 else True

summary = (
    "Summary (timestamps shown in Asia/Shanghai UTC+8):\n\n"
    "metric.csv:\n"
    f"- Total rows: {metric_tot}\n"
    f"- Unique cmdb_id count: {metric_unique}\n"
    f"- Time range: {metric_min} to {metric_max}\n"
    f"- Top cmdb_ids (top 10 by rows): {metric_top_cmdb}\n"
    f"- Top KPI names (top 10): {metric_top_kpi}\n\n"
    "trace.csv:\n"
    f"- Total rows: {trace_tot}\n"
    f"- Unique cmdb_id count: {trace_unique}\n"
    f"- Time range: {trace_min} to {trace_max}\n"
    f"- Top cmdb_ids (top 10 by rows): {trace_top_cmdb}\n"
    f"- Top trace names (top 10): {trace_top_name}\n\n"
    "log.csv:\n"
    f"- Total rows: {log_tot}\n"
    f"- Unique cmdb_id count: {log_unique}\n"
    f"- Time range: {log_min} to {log_max}\n"
    f"- Top cmdb_ids (top 10 by rows): {log_top_cmdb}\n"
    f"- Top log names (top 10): {log_top_name}\n\n"
    "error_logs.csv:\n"
    f"- Total rows: {error_tot}\n"
    f"- Unique cmdb_id count: {error_unique}\n"
    f"- Time range: {error_min} to {error_max}\n"
    f"- Sample rows present: {error_sample_exists} (no sample rows shown because file is empty)\n\n"
    "Notes:\n"
    "- metric.csv is the largest file (~320k rows) and spans 19:30–20:30 (UTC+8).\n"
    "- trace and log files cover shorter windows around 20:00–20:30 with tens of thousands / low thousands of rows.\n"
    "- error_logs.csv is empty in this dataset.\n"
)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(   total_rows  unique_cmdb_count             min_timestamp             max_timestamp
0      320182                 59 2022-03-20 19:30:00+08:00 2022-03-20 20:30:00+08:00,           cmdb_id  count
0       adservice  20801
1      adservice2  20313
2     frontend2-0  11956
3      frontend-2  11468
4      frontend-1  11468
..            ...    ...
45         node-3   3440
46         node-1   3440
47         node-4   3348
48  redis-cart2-0    732
49   redis-cart-0    732

[50 rows x 2 columns],                                              kpi_name  count
0   container.node-6.container_network_receive_pac...   2196
1   container.node-6.container_network_transmit_MB...   2196
2   container.node-6.container_network_transmit_pa...   2196
3   container.node-6.container_network_transmit_pa...   2196
4   container.node-6.container_network_receive_err...   2196
5   container.node-6.container_network_receive_pac...   2196
6   container.node-6.container_network_receive_MB....   2196
7   container.node-6.container_network_transmit_er...   2196
8      container.node-6.container_fs_inodes./dev/vda1   2074
9   container.node-6.container_fs_inodes_free./dev...   2074
10  container.node-6.container_fs_io_current./dev/...   2074
11    container.node-6.container_cpu_load_average_10s   2074
12  container.node-6.container_fs_io_time_weighted...   2074
13  container.node-6.container_fs_io_time_seconds....   2074
14  container.node-6.container_fs_writes_merged./d...   2074
15  container.node-6.container_memory_failures.hie...   2074
16  container.node-6.container_memory_failures.hie...   2074
17  container.node-6.container_memory_failures.con...   2074
18  container.node-6.container_memory_failures.con...   2074
19   container.node-6.container_fs_limit_MB./dev/vda1   2074,    total_rows  unique_cmdb_count             min_timestamp             max_timestamp
0       33640                 40 2022-03-20 20:00:00+08:00 2022-03-20 20:29:00+08:00,               cmdb_id  count
0          frontend-1   2748
1          frontend-2   2540
2          frontend-0   2508
3   checkoutservice-1   2236
4   checkoutservice-2   2212
..                ...    ...
35   paymentservice-1    220
36     emailservice-1    216
37       adservice2-0    120
38    emailservice2-0    120
39  paymentservice2-0    120

[40 rows x 2 columns],                                     trace_name  count
0              trace.from_frontend-1.row_count    627
1             trace.from_frontend-1.error_rate    627
2           trace.from_frontend-1.duration_p95    627
3          trace.from_frontend-1.duration_mean    627
4              trace.from_frontend-2.row_count    575
5             trace.from_frontend-2.error_rate    575
6           trace.from_frontend-2.duration_p95    575
7          trace.from_frontend-2.duration_mean    575
8          trace.from_frontend-0.duration_mean    567
9           trace.from_frontend-0.duration_p95    567
10            trace.from_frontend-0.error_rate    567
11             trace.from_frontend-0.row_count    567
12                    trace.self.duration_mean    480
13                     trace.self.duration_p95    480
14                       trace.self.error_rate    480
15                        trace.self.row_count    480
16  trace.from_checkoutservice-1.duration_mean    446
17   trace.from_checkoutservice-1.duration_p95    446
18     trace.from_checkoutservice-1.error_rate    446
19      trace.from_checkoutservice-1.row_count    446,    total_rows  unique_cmdb_count             min_timestamp             max_timestamp
0        1826                 31 2022-03-20 20:00:00+08:00 2022-03-20 20:30:00+08:00,                     cmdb_id  count
0             cartservice-2     62
1             cartservice-1     62
2             cartservice-0     62
3         currencyservice-0     62
4         currencyservice-2     62
..                      ...    ...
26  recommendationservice-0     60
27        shippingservice-1     60
28         paymentservice-2     54
29         paymentservice-1     54
30             redis-cart-0     12

[31 rows x 2 columns],           log_name  count
0  log.error_count    913
1    log.row_count    913,    total_rows  unique_cmdb_count min_timestamp max_timestamp
0           0                  0           NaT           NaT, Empty DataFrame
Columns: [cmdb_id, count]
Index: [], Empty DataFrame
Columns: [message, count]
Index: [], Empty DataFrame
Columns: [timestamp(Asia/Shanghai), cmdb_id, message]
Index: [])

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[2]:

import pandas as pd
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Reuse existing `metric` if present; otherwise load (defensive)
try:
    metric
except NameError:
    metric = pd.read_csv('metric.csv')

# Ensure correct types and timezone-aware timestamps (UTC+8)
metric['timestamp'] = pd.to_numeric(metric['timestamp'], errors='coerce')
metric['value'] = pd.to_numeric(metric['value'], errors='coerce')
if '_ts_dt' not in metric.columns:
    metric['_ts_dt'] = pd.to_datetime(metric['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
else:
    # ensure _ts_dt is timezone-aware in Asia/Shanghai
    metric['_ts_dt'] = pd.to_datetime(metric['timestamp'], unit='s', utc=True).dt.tz_convert(tz)

# Group by (cmdb_id, kpi_name) and compute required aggregations across the full series
g = metric.groupby(['cmdb_id', 'kpi_name'])
val_agg = g['value'].agg(count='count', mean='mean', p95=lambda x: x.quantile(0.95), p5=lambda x: x.quantile(0.05))
ts_agg = g['_ts_dt'].agg(min_ts='min', max_ts='max')

group_summary = val_agg.join(ts_agg).reset_index()

# Format timestamp columns as compact timezone-aware strings
group_summary['min_timestamp'] = group_summary['min_ts'].astype(str)
group_summary['max_timestamp'] = group_summary['max_ts'].astype(str)

# Select and order requested columns
group_summary = group_summary[['cmdb_id', 'kpi_name', 'count', 'min_timestamp', 'max_timestamp', 'p95', 'p5', 'mean']]

# Order by count desc and limit to top 30 groups
group_summary = group_summary.sort_values(by='count', ascending=False).head(30).reset_index(drop=True)

# Keep the final displayed output compact
group_summary

```
Out[2]:
```
```python
# Summarize the previously computed group_summary (top 30 groups by count)
try:
    group_summary
except NameError:
    import pandas as pd, pytz
    tz = pytz.timezone('Asia/Shanghai')
    metric = pd.read_csv('metric.csv')
    metric['timestamp'] = pd.to_numeric(metric['timestamp'], errors='coerce')
    metric['value'] = pd.to_numeric(metric['value'], errors='coerce')
    metric['_ts_dt'] = pd.to_datetime(metric['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
    g = metric.groupby(['cmdb_id', 'kpi_name'])
    val_agg = g['value'].agg(count='count', mean='mean', p95=lambda x: x.quantile(0.95), p5=lambda x: x.quantile(0.05))
    ts_agg = g['_ts_dt'].agg(min_ts='min', max_ts='max')
    group_summary = val_agg.join(ts_agg).reset_index()
    group_summary['min_timestamp'] = group_summary['min_ts'].astype(str)
    group_summary['max_timestamp'] = group_summary['max_ts'].astype(str)
    group_summary = group_summary[['cmdb_id','kpi_name','count','min_timestamp','max_timestamp','p95','p5','mean']]
    group_summary = group_summary.sort_values('count', ascending=False).head(30).reset_index(drop=True)

# Build a compact plain-English summary
n_groups = group_summary.shape[0]
overall_min = group_summary['min_timestamp'].min()
overall_max = group_summary['max_timestamp'].max()
counts_unique = sorted(group_summary['count'].unique())
counts_summary = (f"all groups have the same count={counts_unique[0]}" 
                  if len(counts_unique)==1 else f"counts vary, sample unique counts: {counts_unique[:5]}")

# Top 3 groups details (compact)
top3_lines = []
for i, row in group_summary.head(3).iterrows():
    top3_lines.append(
        f"{i+1}. {row['cmdb_id']} | {row['kpi_name']} | count={int(row['count'])} | "
        f"p95={row['p95']:.6g} | p5={row['p5']:.6g} | mean={row['mean']:.6g}"
    )

summary = (
    f"Metric grouping summary (top {n_groups} groups by row count):\n"
    f"- Time range across these groups: {overall_min} to {overall_max} (Asia/Shanghai, UTC+8)\n"
    f"- {counts_summary}\n"
    f"- Example top 3 groups:\n  " + "\n  ".join(top3_lines) + "\n"
    f"- Note: Output columns are: cmdb_id, kpi_name, count, min_timestamp, max_timestamp, p95, p5, mean.\n"
)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id                                           kpi_name  count              min_timestamp              max_timestamp           p95            p5          mean
0   shippingservice2-0  mesh.source.shippingservice2.jaeger-collector....     61  2022-03-20 19:30:00+08:00  2022-03-20 20:30:00+08:00  3.631750e+03  2.985000e+03  3.353730e+03
1            adservice                                     app.grpc.count     61  2022-03-20 19:30:00+08:00  2022-03-20 20:30:00+08:00  2.970000e+02  2.480000e+02  2.716885e+02
2            adservice                                       app.grpc.mrt     61  2022-03-20 19:30:00+08:00  2022-03-20 20:30:00+08:00  2.666176e-03  2.437809e-03  2.565834e-03
3            adservice                                        app.grpc.rr     61  2022-03-20 19:30:00+08:00  2022-03-20 20:30:00+08:00  1.000000e+02  1.000000e+02  1.000000e+02
4            adservice                                        app.grpc.sr     61  2022-03-20 19:30:00+08:00  2022-03-20 20:30:00+08:00  1.000000e+02  1.000000e+02  1.000000e+02
..                 ...                                                ...    ...                        ...                        ...           ...           ...           ...
25  shippingservice2-0              container.node-6.container_memory_rss     61  2022-03-20 19:30:00+08:00  2022-03-20 20:30:00+08:00  2.208768e+07  1.171046e+07  1.373134e+07
26  shippingservice2-0             container.node-6.container_memory_swap     61  2022-03-20 19:30:00+08:00  2022-03-20 20:30:00+08:00  0.000000e+00  0.000000e+00  0.000000e+00
27  shippingservice2-0         container.node-6.container_memory_usage_MB     61  2022-03-20 19:30:00+08:00  2022-03-20 20:30:00+08:00  5.210742e+01  4.545312e+01  4.643401e+01
28  shippingservice2-0   container.node-6.container_memory_working_set_MB     61  2022-03-20 19:30:00+08:00  2022-03-20 20:30:00+08:00  4.368164e+01  3.499219e+01  3.674565e+01
29  shippingservice2-0     container.node-6.container_fs_writes./dev/vda1     61  2022-03-20 19:30:00+08:00  2022-03-20 20:30:00+08:00  0.000000e+00  0.000000e+00  0.000000e+00

[30 rows x 8 columns]

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[3]:

import pandas as pd
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Reuse metric if present, otherwise load
try:
    metric
except NameError:
    metric = pd.read_csv('metric.csv')

# Ensure numeric types and timezone-aware timestamp column (Asia/Shanghai)
metric['timestamp'] = pd.to_numeric(metric['timestamp'], errors='coerce')
metric['value'] = pd.to_numeric(metric['value'], errors='coerce')
metric['_ts_dt'] = pd.to_datetime(metric['timestamp'], unit='s', utc=True).dt.tz_convert(tz)

# Build allowed cmdb_id list from task description (nodes, pods, services)
nodes = [f"node-{i}" for i in range(1,7)]
pods = [
    "frontend-0","frontend-1","frontend-2","frontend2-0",
    "shippingservice-0","shippingservice-1","shippingservice-2","shippingservice2-0",
    "checkoutservice-0","checkoutservice-1","checkoutservice-2","checkoutservice2-0",
    "currencyservice-0","currencyservice-1","currencyservice-2","currencyservice2-0",
    "adservice-0","adservice-1","adservice-2","adservice2-0",
    "emailservice-0","emailservice-1","emailservice-2","emailservice2-0",
    "cartservice-0","cartservice-1","cartservice-2","cartservice2-0",
    "productcatalogservice-0","productcatalogservice-1","productcatalogservice-2","productcatalogservice2-0",
    "recommendationservice-0","recommendationservice-1","recommendationservice-2","recommendationservice2-0",
    "paymentservice-0","paymentservice-1","paymentservice-2","paymentservice2-0"
]
services = [
    "frontend","shippingservice","checkoutservice","currencyservice","adservice",
    "emailservice","cartservice","productcatalogservice","recommendationservice","paymentservice"
]
allowed_cmdb = set(nodes + pods + services)

# (1) Compute global P95 and P5 per (cmdb_id, kpi_name) across full series (do BEFORE filtering)
grp = metric.groupby(['cmdb_id', 'kpi_name'])['value']
thresholds = grp.agg(p95=lambda x: x.quantile(0.95), p5=lambda x: x.quantile(0.05)).reset_index()

# (2) Filter metric data to incident window inclusive (Asia/Shanghai)
start_dt = tz.localize(pd.to_datetime("2022-03-20 20:00:00"))
end_dt = tz.localize(pd.to_datetime("2022-03-20 20:30:00"))
metric_window = metric[(metric['_ts_dt'] >= start_dt) & (metric['_ts_dt'] <= end_dt)].copy()

# (3) Restrict to allowed cmdb_id values
metric_window = metric_window[metric_window['cmdb_id'].isin(allowed_cmdb)].copy()

# If nothing in window, produce empty DataFrame with required columns
cols_out = [
    'cmdb_id','kpi_name','count_in_window','count_high','earliest_high_timestamp',
    'count_low','earliest_low_timestamp','max_value_in_window','min_value_in_window','p95','p5'
]

if metric_window.shape[0] == 0:
    result_df = pd.DataFrame(columns=cols_out)
else:
    # Join thresholds into windowed rows so we can mark highs/lows using global thresholds
    mw = metric_window.merge(thresholds, on=['cmdb_id','kpi_name'], how='left')

    # Determine flags
    mw['is_high'] = mw['value'] >= mw['p95']
    mw['is_low'] = mw['value'] <= mw['p5']

    # Aggregations per group
    def agg_group(df):
        count_in_window = int(df.shape[0])
        count_high = int(df['is_high'].sum())
        count_low = int(df['is_low'].sum())
        # earliest timestamps for high/low (as Asia/Shanghai strings) or None
        if count_high > 0:
            earliest_high = df.loc[df['is_high'], '_ts_dt'].min().tz_convert(tz)
            earliest_high = earliest_high.isoformat()
        else:
            earliest_high = pd.NaT
        if count_low > 0:
            earliest_low = df.loc[df['is_low'], '_ts_dt'].min().tz_convert(tz)
            earliest_low = earliest_low.isoformat()
        else:
            earliest_low = pd.NaT
        max_value_in_window = df['value'].max()
        min_value_in_window = df['value'].min()
        # p95/p5 are identical per-row in group; take first non-null
        p95 = df['p95'].iloc[0]
        p5 = df['p5'].iloc[0]
        return pd.Series({
            'count_in_window': count_in_window,
            'count_high': count_high,
            'earliest_high_timestamp': earliest_high,
            'count_low': count_low,
            'earliest_low_timestamp': earliest_low,
            'max_value_in_window': max_value_in_window,
            'min_value_in_window': min_value_in_window,
            'p95': p95,
            'p5': p5
        })

    grouped_window = mw.groupby(['cmdb_id','kpi_name']).apply(agg_group).reset_index()

    # Compute total anomalies and filter/sort
    grouped_window['total_anomalies'] = grouped_window['count_high'] + grouped_window['count_low']

    # Keep only groups with at least one point in window (count_in_window>0) - already ensured by grouping on windowed data
    result_df = grouped_window.sort_values(by='total_anomalies', ascending=False).head(20)

    # Reorder columns as requested (drop helper total_anomalies)
    result_df = result_df[[
        'cmdb_id','kpi_name','count_in_window','count_high','earliest_high_timestamp',
        'count_low','earliest_low_timestamp','max_value_in_window','min_value_in_window','p95','p5'
    ]].reset_index(drop=True)

# Final compact output (top 20)
result_df.head(20)

```
Out[3]:
```
```python
# Build a concise plain-English summary from the last computed `result_df`
try:
    result_df
except NameError:
    # If result_df is not present, load minimal context and produce an empty summary
    result_df = None

if result_df is None or result_df.shape[0] == 0:
    summary = "No metric groups found in the incident window for the specified root-cause component list."
else:
    n_groups = result_df.shape[0]
    window_start = "2022-03-20 20:00:00+08:00"
    window_end = "2022-03-20 20:30:00+08:00"
    # Prepare top 3 example lines
    top_lines = []
    for i, row in result_df.head(3).iterrows():
        top_lines.append(
            f"{i+1}. {row['cmdb_id']} | {row['kpi_name']} | count_in_window={int(row['count_in_window'])} "
            f"| count_high={int(row['count_high'])} | count_low={int(row['count_low'])} "
            f"| earliest_high={row['earliest_high_timestamp']} | earliest_low={row['earliest_low_timestamp']} "
            f"| p95={row['p95']:.6g} | p5={row['p5']:.6g}"
        )
    summary = (
        f"Analysis restricted to allowed components, window {window_start} through {window_end} (Asia/Shanghai UTC+8):\n"
        f"- Groups returned (top by anomalies): {n_groups} (showing top 20 limited by request).\n"
        f"- Most groups in the top results are for cmdb_id 'shippingservice2-0' and 'adservice'.\n"
        f"- Many groups show count_in_window=31 (full 31-minute window) and count_high=count_low=31, meaning\n"
        f"  values in-window equal the precomputed global P95 and P5 (thresholds identical to observed values).\n"
        f"- Example top 3 groups:\n  " + "\n  ".join(top_lines) + "\n"
        f"- Columns provided per group: cmdb_id, kpi_name, count_in_window, count_high, earliest_high_timestamp,\n"
        f"  count_low, earliest_low_timestamp, max_value_in_window, min_value_in_window, p95, p5.\n"
        f"- Interpretation note: identical p95 and p5 (or equal to observed values) suggests constant or degenerate\n"
        f"  series for those KPIs across the full period; investigate those KPIs and nodes for potential instrumentation or\n"
        f"  metric-collection issues if these look suspicious."
    )

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id                                           kpi_name  count_in_window  count_high    earliest_high_timestamp  count_low     earliest_low_timestamp  max_value_in_window  min_value_in_window           p95            p5
0   shippingservice2-0     container.node-6.container_memory_max_usage_MB               31          31  2022-03-20T20:00:00+08:00         31  2022-03-20T20:00:00+08:00         1.287266e+02         1.287266e+02  1.287266e+02  1.287266e+02
1   shippingservice2-0             container.node-6.container_memory_swap               31          31  2022-03-20T20:00:00+08:00         31  2022-03-20T20:00:00+08:00         0.000000e+00         0.000000e+00  0.000000e+00  0.000000e+00
2   shippingservice2-0     container.node-6.container_fs_writes./dev/vda1               31          31  2022-03-20T20:00:00+08:00         31  2022-03-20T20:00:00+08:00         0.000000e+00         0.000000e+00  0.000000e+00  0.000000e+00
3   shippingservice2-0  container.node-6.container_fs_writes_merged./d...               31          31  2022-03-20T20:00:00+08:00         31  2022-03-20T20:00:00+08:00         0.000000e+00         0.000000e+00  0.000000e+00  0.000000e+00
4   shippingservice2-0          container.node-6.container_memory_failcnt               31          31  2022-03-20T20:00:00+08:00         31  2022-03-20T20:00:00+08:00         0.000000e+00         0.000000e+00  0.000000e+00  0.000000e+00
5   shippingservice2-0      container.node-6.container_fs_reads./dev/vda1               31          31  2022-03-20T20:00:00+08:00         31  2022-03-20T20:00:00+08:00         0.000000e+00         0.000000e+00  0.000000e+00  0.000000e+00
6   shippingservice2-0  container.node-6.container_fs_reads_merged./de...               31          31  2022-03-20T20:00:00+08:00         31  2022-03-20T20:00:00+08:00         0.000000e+00         0.000000e+00  0.000000e+00  0.000000e+00
7   shippingservice2-0  container.node-6.container_fs_sector_reads./de...               31          31  2022-03-20T20:00:00+08:00         31  2022-03-20T20:00:00+08:00         0.000000e+00         0.000000e+00  0.000000e+00  0.000000e+00
8   shippingservice2-0  container.node-6.container_fs_sector_writes./d...               31          31  2022-03-20T20:00:00+08:00         31  2022-03-20T20:00:00+08:00         0.000000e+00         0.000000e+00  0.000000e+00  0.000000e+00
9   shippingservice2-0  container.node-6.container_fs_write_seconds./d...               31          31  2022-03-20T20:00:00+08:00         31  2022-03-20T20:00:00+08:00         0.000000e+00         0.000000e+00  0.000000e+00  0.000000e+00
10  shippingservice2-0     container.node-6.container_fs_inodes./dev/vda1               31          31  2022-03-20T20:00:00+08:00         31  2022-03-20T20:00:00+08:00         0.000000e+00         0.000000e+00  0.000000e+00  0.000000e+00
11  shippingservice2-0  container.node-6.container_fs_inodes_free./dev...               31          31  2022-03-20T20:00:00+08:00         31  2022-03-20T20:00:00+08:00         0.000000e+00         0.000000e+00  0.000000e+00  0.000000e+00
12  shippingservice2-0  container.node-6.container_fs_io_current./dev/...               31          31  2022-03-20T20:00:00+08:00         31  2022-03-20T20:00:00+08:00         0.000000e+00         0.000000e+00  0.000000e+00  0.000000e+00
13  shippingservice2-0  container.node-6.container_fs_io_time_seconds....               31          31  2022-03-20T20:00:00+08:00         31  2022-03-20T20:00:00+08:00         0.000000e+00         0.000000e+00  0.000000e+00  0.000000e+00
14  shippingservice2-0  container.node-6.container_fs_io_time_weighted...               31          31  2022-03-20T20:00:00+08:00         31  2022-03-20T20:00:00+08:00         0.000000e+00         0.000000e+00  0.000000e+00  0.000000e+00
15  shippingservice2-0   container.node-6.container_fs_limit_MB./dev/vda1               31          31  2022-03-20T20:00:00+08:00         31  2022-03-20T20:00:00+08:00         6.046307e+05         6.046307e+05  6.046307e+05  6.046307e+05
16  shippingservice2-0  container.node-6.container_fs_read_seconds./de...               31          31  2022-03-20T20:00:00+08:00         31  2022-03-20T20:00:00+08:00         0.000000e+00         0.000000e+00  0.000000e+00  0.000000e+00
17           adservice  runtime.java_lang_GarbageCollector_LastGcInfo_...               31          31  2022-03-20T20:00:00+08:00         31  2022-03-20T20:00:00+08:00        -1.000000e+00        -1.000000e+00 -1.000000e+00 -1.000000e+00
18           adservice  runtime.java_lang_GarbageCollector_LastGcInfo_...               31          31  2022-03-20T20:00:00+08:00         31  2022-03-20T20:00:00+08:00        -1.000000e+00        -1.000000e+00 -1.000000e+00 -1.000000e+00
19           adservice  runtime.java_lang_GarbageCollector_LastGcInfo_...               31          31  2022-03-20T20:00:00+08:00         31  2022-03-20T20:00:00+08:00         5.242880e+06         5.242880e+06  5.242880e+06  5.242880e+06```
```

In [ ]:
# In[4]:

import pandas as pd
import numpy as np
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Reuse existing metric and thresholds if present, otherwise load/compute
try:
    metric
except NameError:
    metric = pd.read_csv('metric.csv')

metric['timestamp'] = pd.to_numeric(metric['timestamp'], errors='coerce')
metric['value'] = pd.to_numeric(metric['value'], errors='coerce')
metric['_ts_dt'] = pd.to_datetime(metric['timestamp'], unit='s', utc=True).dt.tz_convert(tz)

# Allowed components (nodes, pods, services)
nodes = [f"node-{i}" for i in range(1,7)]
pods = [
    "frontend-0","frontend-1","frontend-2","frontend2-0",
    "shippingservice-0","shippingservice-1","shippingservice-2","shippingservice2-0",
    "checkoutservice-0","checkoutservice-1","checkoutservice-2","checkoutservice2-0",
    "currencyservice-0","currencyservice-1","currencyservice-2","currencyservice2-0",
    "adservice-0","adservice-1","adservice-2","adservice2-0",
    "emailservice-0","emailservice-1","emailservice-2","emailservice2-0",
    "cartservice-0","cartservice-1","cartservice-2","cartservice2-0",
    "productcatalogservice-0","productcatalogservice-1","productcatalogservice-2","productcatalogservice2-0",
    "recommendationservice-0","recommendationservice-1","recommendationservice-2","recommendationservice2-0",
    "paymentservice-0","paymentservice-1","paymentservice-2","paymentservice2-0"
]
services = [
    "frontend","shippingservice","checkoutservice","currencyservice","adservice",
    "emailservice","cartservice","productcatalogservice","recommendationservice","paymentservice"
]
allowed_cmdb = set(nodes + pods + services)

# (1) Global thresholds p95/p5 per (cmdb_id,kpi_name) from full series (reuse if available)
try:
    thresholds
except NameError:
    grp = metric.groupby(['cmdb_id', 'kpi_name'])['value']
    thresholds = grp.agg(p95=lambda x: x.quantile(0.95), p5=lambda x: x.quantile(0.05)).reset_index()

# (2) Filter to incident window inclusive (Asia/Shanghai)
start_dt = tz.localize(pd.to_datetime("2022-03-20 20:00:00"))
end_dt = tz.localize(pd.to_datetime("2022-03-20 20:30:00"))
metric_window = metric[(metric['_ts_dt'] >= start_dt) & (metric['_ts_dt'] <= end_dt)].copy()

# (3) Restrict to allowed cmdb_id set
metric_window = metric_window[metric_window['cmdb_id'].isin(allowed_cmdb)].copy()

# Prepare output columns
out_cols = [
    'cmdb_id','kpi_name','count_in_window','mean_in_window','max_value_in_window','min_value_in_window',
    'p95','p5','positive_breach_count','earliest_positive_breach_timestamp',
    'negative_breach_count','earliest_negative_breach_timestamp','relative_breach'
]

if metric_window.shape[0] == 0:
    result_top20 = pd.DataFrame(columns=out_cols)
else:
    # Merge global thresholds into windowed data
    mw = metric_window.merge(thresholds, on=['cmdb_id','kpi_name'], how='left')

    # Strict breaches: > p95 and < p5
    mw['is_pos'] = mw['value'] > mw['p95']
    mw['is_neg'] = mw['value'] < mw['p5']

    mw['ts_pos'] = mw['_ts_dt'].where(mw['is_pos'])
    mw['ts_neg'] = mw['_ts_dt'].where(mw['is_neg'])

    # Aggregate per group
    agg = mw.groupby(['cmdb_id','kpi_name']).agg(
        count_in_window = ('value','count'),
        mean_in_window  = ('value','mean'),
        max_value_in_window = ('value','max'),
        min_value_in_window = ('value','min'),
        p95 = ('p95','first'),
        p5  = ('p5','first'),
        positive_breach_count = ('is_pos','sum'),
        earliest_positive_breach_timestamp = ('ts_pos','min'),
        negative_breach_count = ('is_neg','sum'),
        earliest_negative_breach_timestamp = ('ts_neg','min')
    ).reset_index()

    # Convert earliest timestamps to ISO strings (Asia/Shanghai) or NaT
    for col in ['earliest_positive_breach_timestamp','earliest_negative_breach_timestamp']:
        agg[col] = pd.to_datetime(agg[col], utc=True, errors='coerce')  # coerces None->NaT, keeps tz-aware if any
        agg[col] = agg[col].dt.tz_convert(tz)
        agg[col] = agg[col].apply(lambda x: x.isoformat() if pd.notnull(x) else pd.NaT)

    # Ensure integer counts
    agg['positive_breach_count'] = agg['positive_breach_count'].astype(int)
    agg['negative_breach_count'] = agg['negative_breach_count'].astype(int)
    agg['count_in_window'] = agg['count_in_window'].astype(int)

    # (4) Compute relative_breach as defined:
    # relative_breach = max( (max - p95)/max(|p95|,1e-9), (p5 - min)/max(|p5|,1e-9) )
    denom_p95 = agg['p95'].abs().clip(lower=1e-9)
    denom_p5  = agg['p5'].abs().clip(lower=1e-9)

    rel_up = (agg['max_value_in_window'] - agg['p95']) / denom_p95
    rel_down = (agg['p5'] - agg['min_value_in_window']) / denom_p5

    # Replace NaN with 0
    rel_up = rel_up.fillna(0)
    rel_down = rel_down.fillna(0)

    agg['relative_breach'] = np.maximum(rel_up, rel_down)

    # (5) Select top 20 by relative_breach desc and output requested columns
    result_top20 = agg[[
        'cmdb_id','kpi_name','count_in_window','mean_in_window','max_value_in_window','min_value_in_window',
        'p95','p5','positive_breach_count','earliest_positive_breach_timestamp',
        'negative_breach_count','earliest_negative_breach_timestamp','relative_breach'
    ]].sort_values(by='relative_breach', ascending=False).head(20).reset_index(drop=True)

# Final compact result
result_top20

```
Out[4]:
```
```python
# Compose a concise plain-English summary from the last computed `result_top20`
try:
    result_top20
except NameError:
    result_top20 = pd.DataFrame()

if result_top20.empty:
    summary = "No results available for the requested analysis."
else:
    # Top 5 groups brief lines
    lines = []
    for i, row in result_top20.head(5).iterrows():
        lines.append(
            f"{i+1}. {row['cmdb_id']} | {row['kpi_name']} | count_in_window={int(row['count_in_window'])} "
            f"| pos_breach={int(row['positive_breach_count'])} (earliest={row['earliest_positive_breach_timestamp']}) "
            f"| neg_breach={int(row['negative_breach_count'])} (earliest={row['earliest_negative_breach_timestamp']}) "
            f"| p95={row['p95']:.6g} p5={row['p5']:.6g} | relative_breach={row['relative_breach']:.6g}"
        )

    summary = (
        "Summary of metric breaches (window 2022-03-20 20:00:00 to 20:30:00 Asia/Shanghai):\n\n"
        f"- Returned groups: {len(result_top20)} (top 20 by relative_breach).\n"
        "- Key observations:\n"
        "  1) The highest-ranked groups are mostly memory-failure and FS-related KPIs on several components\n"
        "     (e.g., productcatalogservice2-0, checkoutservice-0, shippingservice2-0, frontend2-0).\n"
        "  2) Nearly all top groups have full coverage in-window (count_in_window=31).\n"
        "  3) positive_breach_count is small (1–3) for these groups; negative_breach_count is 0 for the top results.\n"
        "  4) Many p95/p5 values equal 0 (or extremely small), producing very large relative_breach scores when\n"
        "     any nonzero max/min appears in-window. This can inflate the relative_breach metric — check for\n"
        "     degenerate thresholds or instrumentation issues before concluding severity.\n\n"
        "Example top groups:\n  " + "\n  ".join(lines) + "\n\n"
        "Interpretation guidance:\n"
        "- Investigate the listed cmdb_id/kpi pairs (start with the top entries) to confirm whether spikes are\n"
        "  real or caused by metric collection anomalies (e.g., p95/p5 == 0). Prioritize memory-failure and\n"
        "  filesystem metrics on the named components for root-cause analysis."
    )

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id                                           kpi_name  count_in_window  mean_in_window  max_value_in_window  min_value_in_window  p95   p5  positive_breach_count earliest_positive_breach_timestamp  negative_breach_count earliest_negative_breach_timestamp  relative_breach
0   productcatalogservice2-0  container.node-6.container_memory_failures.con...               31        1.064516            33.000000                  0.0  0.0  0.0                      1          2022-03-20T20:04:00+08:00                      0                                NaT     3.300000e+10
1   productcatalogservice2-0  container.node-6.container_memory_failures.hie...               31        1.064516            33.000000                  0.0  0.0  0.0                      1          2022-03-20T20:04:00+08:00                      0                                NaT     3.300000e+10
2          checkoutservice-0  container.node-6.container_memory_failures.hie...               31        2.129032            33.000000                  0.0  0.0  0.0                      2          2022-03-20T20:02:00+08:00                      0                                NaT     3.300000e+10
3         shippingservice2-0  container.node-6.container_memory_failures.hie...               31        2.129032            33.000000                  0.0  0.0  0.0                      3          2022-03-20T20:23:00+08:00                      0                                NaT     3.300000e+10
4         shippingservice2-0  container.node-6.container_memory_failures.con...               31        2.129032            33.000000                  0.0  0.0  0.0                      3          2022-03-20T20:23:00+08:00                      0                                NaT     3.300000e+10
5          checkoutservice-0  container.node-6.container_memory_failures.con...               31        2.129032            33.000000                  0.0  0.0  0.0                      2          2022-03-20T20:02:00+08:00                      0                                NaT     3.300000e+10
6                frontend2-0  container.node-6.container_memory_failures.con...               31        1.064516            16.500000                  0.0  0.0  0.0                      2          2022-03-20T20:02:00+08:00                      0                                NaT     1.650000e+10
7                frontend2-0  container.node-6.container_memory_failures.hie...               31        1.064516            16.500000                  0.0  0.0  0.0                      2          2022-03-20T20:02:00+08:00                      0                                NaT     1.650000e+10
8         shippingservice2-0      container.node-6.container_fs_writes./dev/vda               31        0.354839             8.000000                  0.0  0.0  0.0                      3          2022-03-20T20:23:00+08:00                      0                                NaT     8.000000e+09
9          shippingservice-2       container.node-5.container_fs_reads./dev/vda               31        0.354839             6.000000                  0.0  0.0  0.0                      2          2022-03-20T20:14:00+08:00                      0                                NaT     6.000000e+09
10                frontend-1       container.node-5.container_fs_reads./dev/vda               31        0.064516             1.333333                  0.0  0.0  0.0                      2          2022-03-20T20:02:00+08:00                      0                                NaT     1.333333e+09
11                    node-4                          system.process.zombie.num               31        0.032258             1.000000                  0.0  0.0  0.0                      1          2022-03-20T20:06:00+08:00                      0                                NaT     1.000000e+09
12         currencyservice-1  container.node-6.container_cpu_cfs_throttled_p...               31        0.032258             1.000000                  0.0  0.0  0.0                      1          2022-03-20T20:11:00+08:00                      0                                NaT     1.000000e+09
13             cartservice-1  container.node-6.container_cpu_cfs_throttled_p...               31        0.032258             0.500000                  0.0  0.0  0.0                      2          2022-03-20T20:07:00+08:00                      0                                NaT     5.000000e+08
14  recommendationservice2-0  container.node-6.container_cpu_cfs_throttled_p...               31        0.032258             0.500000                  0.0  0.0  0.0                      2          2022-03-20T20:26:00+08:00                      0                                NaT     5.000000e+08
15         shippingservice-2    container.node-5.container_fs_reads_MB./dev/vda               31        0.006300             0.128906                  0.0  0.0  0.0                      2          2022-03-20T20:14:00+08:00                      0                                NaT     1.289062e+08
16        shippingservice2-0   container.node-6.container_fs_writes_MB./dev/vda               31        0.003654             0.085938                  0.0  0.0  0.0                      3          2022-03-20T20:23:00+08:00                      0                                NaT     8.593750e+07
17         currencyservice-1  container.node-6.container_cpu_cfs_throttled_s...               31        0.001414             0.043819                  0.0  0.0  0.0                      1          2022-03-20T20:11:00+08:00                      0                                NaT     4.381934e+07
18             cartservice-1  container.node-6.container_cpu_cfs_throttled_s...               31        0.001642             0.025453                  0.0  0.0  0.0                      2          2022-03-20T20:07:00+08:00                      0                                NaT     2.545263e+07
19  recommendationservice2-0  container.node-6.container_cpu_cfs_throttled_s...               31        0.001576             0.024420                  0.0  0.0  0.0                      2          2022-03-20T20:26:00+08:00                      0                                NaT     2.442044e+07```
```

In [ ]:
# In[5]:

import pandas as pd
import numpy as np
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Reuse trace if present, otherwise load
try:
    trace
except NameError:
    trace = pd.read_csv('trace.csv')

# Ensure numeric types and add timezone-aware timestamp in Asia/Shanghai
trace['timestamp'] = pd.to_numeric(trace['timestamp'], errors='coerce')
trace['value'] = pd.to_numeric(trace['value'], errors='coerce')
trace['_ts_dt'] = pd.to_datetime(trace['timestamp'], unit='s', utc=True).dt.tz_convert(tz)

# Reuse allowed_cmdb set if available, otherwise define it (nodes, pods, services)
try:
    allowed_cmdb
except NameError:
    nodes = [f"node-{i}" for i in range(1,7)]
    pods = [
        "frontend-0","frontend-1","frontend-2","frontend2-0",
        "shippingservice-0","shippingservice-1","shippingservice-2","shippingservice2-0",
        "checkoutservice-0","checkoutservice-1","checkoutservice-2","checkoutservice2-0",
        "currencyservice-0","currencyservice-1","currencyservice-2","currencyservice2-0",
        "adservice-0","adservice-1","adservice-2","adservice2-0",
        "emailservice-0","emailservice-1","emailservice-2","emailservice2-0",
        "cartservice-0","cartservice-1","cartservice-2","cartservice2-0",
        "productcatalogservice-0","productcatalogservice-1","productcatalogservice-2","productcatalogservice2-0",
        "recommendationservice-0","recommendationservice-1","recommendationservice-2","recommendationservice2-0",
        "paymentservice-0","paymentservice-1","paymentservice-2","paymentservice2-0"
    ]
    services = [
        "frontend","shippingservice","checkoutservice","currencyservice","adservice",
        "emailservice","cartservice","productcatalogservice","recommendationservice","paymentservice"
    ]
    allowed_cmdb = set(nodes + pods + services)

# (1) Compute global P95 and P5 per (cmdb_id, trace_name) using full series (do NOT recompute later)
trace_grp = trace.groupby(['cmdb_id', 'trace_name'])['value']
trace_thresholds = trace_grp.agg(p95=lambda x: x.quantile(0.95), p5=lambda x: x.quantile(0.05)).reset_index()

# (2) Filter to incident window inclusive (Asia/Shanghai)
start_dt = tz.localize(pd.to_datetime("2022-03-20 20:00:00"))
end_dt = tz.localize(pd.to_datetime("2022-03-20 20:30:00"))
trace_window = trace[(trace['_ts_dt'] >= start_dt) & (trace['_ts_dt'] <= end_dt)].copy()

# (3) Restrict to allowed cmdb_id values
trace_window = trace_window[trace_window['cmdb_id'].isin(allowed_cmdb)].copy()

# Prepare output columns
out_cols = [
    'cmdb_id','trace_name','count_in_window','count_high','earliest_high_timestamp',
    'count_low','earliest_low_timestamp','max_value_in_window','min_value_in_window','p95','p5','relative_breach'
]

if trace_window.shape[0] == 0:
    trace_result_top20 = pd.DataFrame(columns=out_cols)
else:
    # (1 reused) Merge global thresholds into the windowed rows
    tw = trace_window.merge(trace_thresholds, on=['cmdb_id','trace_name'], how='left')

    # Strict breaches: > p95 and < p5
    tw['is_high'] = tw['value'] > tw['p95']
    tw['is_low'] = tw['value'] < tw['p5']
    tw['ts_high'] = tw['_ts_dt'].where(tw['is_high'])
    tw['ts_low'] = tw['_ts_dt'].where(tw['is_low'])

    # Aggregate per (cmdb_id, trace_name)
    agg = tw.groupby(['cmdb_id','trace_name']).agg(
        count_in_window = ('value','count'),
        count_high = ('is_high','sum'),
        earliest_high_timestamp = ('ts_high','min'),
        count_low = ('is_low','sum'),
        earliest_low_timestamp = ('ts_low','min'),
        max_value_in_window = ('value','max'),
        min_value_in_window = ('value','min'),
        p95 = ('p95','first'),
        p5  = ('p5','first')
    ).reset_index()

    # Convert earliest timestamps to ISO strings in Asia/Shanghai or NaT
    for col in ['earliest_high_timestamp','earliest_low_timestamp']:
        agg[col] = pd.to_datetime(agg[col], utc=True, errors='coerce')
        agg[col] = agg[col].dt.tz_convert(tz)
        agg[col] = agg[col].apply(lambda x: x.isoformat() if pd.notnull(x) else pd.NaT)

    # Ensure integer counts
    agg['count_in_window'] = agg['count_in_window'].astype(int)
    agg['count_high'] = agg['count_high'].astype(int)
    agg['count_low'] = agg['count_low'].astype(int)

    # (4) Compute relative_breach similar to metric analysis
    denom_p95 = agg['p95'].abs().clip(lower=1e-9)
    denom_p5  = agg['p5'].abs().clip(lower=1e-9)

    rel_up = (agg['max_value_in_window'] - agg['p95']) / denom_p95
    rel_down = (agg['p5'] - agg['min_value_in_window']) / denom_p5

    rel_up = rel_up.fillna(0)
    rel_down = rel_down.fillna(0)

    agg['relative_breach'] = np.maximum(rel_up, rel_down)

    # (5) Return top 20 groups ordered by relative_breach desc with requested columns
    trace_result_top20 = agg[[
        'cmdb_id','trace_name','count_in_window','count_high','earliest_high_timestamp',
        'count_low','earliest_low_timestamp','max_value_in_window','min_value_in_window','p95','p5','relative_breach'
    ]].sort_values(by='relative_breach', ascending=False).head(20).reset_index(drop=True)

# Final compact output (top 20)
trace_result_top20

```
Out[5]:
```
```python
# Compose a concise plain-English summary from the most recent trace_result_top20
try:
    trace_result_top20
except NameError:
    trace_result_top20 = pd.DataFrame()

if trace_result_top20.empty:
    summary = "No trace groups found in the incident window for the allowed components."
else:
    lines = []
    for i, row in trace_result_top20.head(5).iterrows():
        lines.append(
            f"{i+1}. {row['cmdb_id']} | {row['trace_name']} | count_in_window={int(row['count_in_window'])} "
            f"| count_high={int(row['count_high'])} (earliest={row['earliest_high_timestamp']}) "
            f"| count_low={int(row['count_low'])} (earliest={row['earliest_low_timestamp']}) "
            f"| p95={row['p95']:.6g} p5={row['p5']:.6g} | relative_breach={row['relative_breach']:.6g}"
        )

    summary = (
        "Trace analysis (window 2022-03-20 20:00:00 to 20:30:00 Asia/Shanghai):\n\n"
        f"- Returned top {len(trace_result_top20)} groups (sorted by relative_breach).\n"
        "- Key findings:\n"
        "  1) Highest relative breaches are observed for cartservice and checkoutservice traces\n"
        "     (e.g., cartservice-0, cartservice-1, checkoutservice-1), followed by frontend↔cart/checkout\n"
        "     interactions.\n"
        "  2) Many top entries have small or zero p95/p5 baselines; a single elevated in-window value\n"
        "     produces very large relative_breach scores. Verify whether p95/p5 == 0 is a valid baseline\n"
        "     or an instrumentation artifact before prioritizing.\n"
        "  3) Earliest high trace anomalies occur at specific minutes that can be correlated with metrics:\n"
        "     see timestamps in the example lines below (times in Asia/Shanghai UTC+8).\n\n"
        "Example top 5 trace groups:\n  " + "\n  ".join(lines) + "\n\n"
        "Suggested next steps:\n"
        "- Correlate the listed earliest_high_timestamp values (e.g., 20:23, 20:19, 20:16, 20:01, 20:00) with\n"
        "  metric spikes from the earlier metric analysis (memory/fs/cpu) to find likely root causes.\n"
        "- Inspect trace payloads and service logs for cartservice and checkoutservice around those minutes.\n"
        "- Confirm trace baseline calculations (p95/p5) to avoid false positives caused by degenerate baselines."
    )

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id                                  trace_name  count_in_window  count_high    earliest_high_timestamp  count_low     earliest_low_timestamp  max_value_in_window  min_value_in_window       p95        p5  relative_breach
0       cartservice-0                     trace.self.duration_p95               30           1  2022-03-20T20:23:00+08:00          0                        NaT             0.001400             0.000000  0.000000  0.000000     1.400000e+06
1       cartservice-1   trace.from_checkoutservice-1.duration_p95               29           1  2022-03-20T20:19:00+08:00          0                        NaT             0.001000             0.000000  0.000000  0.000000     1.000000e+06
2       cartservice-1  trace.from_checkoutservice-1.duration_mean               29           1  2022-03-20T20:19:00+08:00          0                        NaT             0.001000             0.000000  0.000000  0.000000     1.000000e+06
3   checkoutservice-1        trace.to_cartservice-1.duration_mean               29           1  2022-03-20T20:19:00+08:00          0                        NaT             0.001000             0.000000  0.000000  0.000000     1.000000e+06
4   checkoutservice-1         trace.to_cartservice-1.duration_p95               29           1  2022-03-20T20:19:00+08:00          0                        NaT             0.001000             0.000000  0.000000  0.000000     1.000000e+06
5       cartservice-1          trace.from_frontend-0.duration_p95               30           1  2022-03-20T20:28:00+08:00          0                        NaT             0.166650             0.000000  0.001000  0.000000     1.656500e+02
6          frontend-0         trace.to_cartservice-1.duration_p95               30           1  2022-03-20T20:28:00+08:00          0                        NaT             0.166650             0.000000  0.001000  0.000000     1.656500e+02
7          frontend-0        trace.to_cartservice-1.duration_mean               30           2  2022-03-20T20:16:00+08:00          0                        NaT             0.030300             0.000000  0.000294  0.000000     1.021665e+02
8       cartservice-1         trace.from_frontend-0.duration_mean               30           2  2022-03-20T20:16:00+08:00          0                        NaT             0.030300             0.000000  0.000294  0.000000     1.021665e+02
9       cartservice-1                    trace.self.duration_mean               30           2  2022-03-20T20:01:00+08:00          0                        NaT             0.006733             0.000000  0.000095  0.000000     7.013576e+01
10      cartservice-0  trace.from_checkoutservice-1.duration_mean               28           2  2022-03-20T20:00:00+08:00          0                        NaT             0.011000             0.000000  0.000825  0.000000     1.233333e+01
11  checkoutservice-1        trace.to_cartservice-0.duration_mean               28           2  2022-03-20T20:00:00+08:00          0                        NaT             0.011000             0.000000  0.000825  0.000000     1.233333e+01
12  checkoutservice-1         trace.to_cartservice-0.duration_p95               28           2  2022-03-20T20:00:00+08:00          0                        NaT             0.011000             0.000000  0.000982  0.000000     1.019593e+01
13      cartservice-0   trace.from_checkoutservice-1.duration_p95               28           2  2022-03-20T20:00:00+08:00          0                        NaT             0.011000             0.000000  0.000982  0.000000     1.019593e+01
14         frontend-1     trace.to_shippingservice-2.duration_p95               30           2  2022-03-20T20:07:00+08:00          2  2022-03-20T20:08:00+08:00             0.002207             0.000061  0.000269  0.000061     7.209038e+00
15  shippingservice-2          trace.from_frontend-1.duration_p95               30           2  2022-03-20T20:07:00+08:00          2  2022-03-20T20:08:00+08:00             0.002207             0.000061  0.000269  0.000061     7.209038e+00
16      cartservice-0                    trace.self.duration_mean               30           2  2022-03-20T20:18:00+08:00          0                        NaT             0.000298             0.000000  0.000042  0.000000     6.137725e+00
17  checkoutservice-2    trace.to_shippingservice-2.duration_mean               26           2  2022-03-20T20:26:00+08:00          1  2022-03-20T20:10:00+08:00             0.000428             0.000058  0.000080  0.000059     4.350000e+00
18  shippingservice-2  trace.from_checkoutservice-2.duration_mean               26           2  2022-03-20T20:26:00+08:00          1  2022-03-20T20:10:00+08:00             0.000428             0.000058  0.000080  0.000059     4.350000e+00
19  checkoutservice-0     trace.to_currencyservice-0.duration_p95               30           2  2022-03-20T20:01:00+08:00          2  2022-03-20T20:18:00+08:00             0.000745             0.000108  0.000146  0.000114     4.115002e+00```
```

In [ ]:
# In[6]:

import pandas as pd
import numpy as np
import pytz

# Reuse variables if present
tz = pytz.timezone('Asia/Shanghai')
try:
    metric
except NameError:
    metric = pd.read_csv('metric.csv')
try:
    trace
except NameError:
    trace = pd.read_csv('trace.csv')

# Ensure types and timezone-aware timestamps
metric['timestamp'] = pd.to_numeric(metric['timestamp'], errors='coerce')
metric['value'] = pd.to_numeric(metric['value'], errors='coerce')
metric['_ts_dt'] = pd.to_datetime(metric['timestamp'], unit='s', utc=True).dt.tz_convert(tz)

trace['timestamp'] = pd.to_numeric(trace['timestamp'], errors='coerce')
trace['value'] = pd.to_numeric(trace['value'], errors='coerce')
trace['_ts_dt'] = pd.to_datetime(trace['timestamp'], unit='s', utc=True).dt.tz_convert(tz)

# Allowed components (reuse if available)
try:
    allowed_cmdb
except NameError:
    nodes = [f"node-{i}" for i in range(1,7)]
    pods = [
        "frontend-0","frontend-1","frontend-2","frontend2-0",
        "shippingservice-0","shippingservice-1","shippingservice-2","shippingservice2-0",
        "checkoutservice-0","checkoutservice-1","checkoutservice-2","checkoutservice2-0",
        "currencyservice-0","currencyservice-1","currencyservice-2","currencyservice2-0",
        "adservice-0","adservice-1","adservice-2","adservice2-0",
        "emailservice-0","emailservice-1","emailservice-2","emailservice2-0",
        "cartservice-0","cartservice-1","cartservice-2","cartservice2-0",
        "productcatalogservice-0","productcatalogservice-1","productcatalogservice-2","productcatalogservice2-0",
        "recommendationservice-0","recommendationservice-1","recommendationservice-2","recommendationservice2-0",
        "paymentservice-0","paymentservice-1","paymentservice-2","paymentservice2-0"
    ]
    services = [
        "frontend","shippingservice","checkoutservice","currencyservice","adservice",
        "emailservice","cartservice","productcatalogservice","recommendationservice","paymentservice"
    ]
    allowed_cmdb = set(nodes + pods + services)

# (1) Compute / reuse global thresholds for metric and trace across full series (must be global BEFORE filtering)
try:
    thresholds  # from previous steps
except NameError:
    thresholds = metric.groupby(['cmdb_id','kpi_name'])['value'].agg(
        p95=lambda x: x.quantile(0.95), p5=lambda x: x.quantile(0.05)
    ).reset_index()

try:
    trace_thresholds
except NameError:
    trace_thresholds = trace.groupby(['cmdb_id','trace_name'])['value'].agg(
        p95=lambda x: x.quantile(0.95), p5=lambda x: x.quantile(0.05)
    ).reset_index()

# (2) Incident window filter (inclusive)
start_dt = tz.localize(pd.to_datetime("2022-03-20 20:00:00"))
end_dt = tz.localize(pd.to_datetime("2022-03-20 20:30:00"))

metric_window = metric[(metric['_ts_dt'] >= start_dt) & (metric['_ts_dt'] <= end_dt)].copy()
trace_window  = trace[(trace['_ts_dt']  >= start_dt) & (trace['_ts_dt']  <= end_dt)].copy()

# (3) Restrict to allowed components
metric_window = metric_window[metric_window['cmdb_id'].isin(allowed_cmdb)].copy()
trace_window  = trace_window[trace_window['cmdb_id'].isin(allowed_cmdb)].copy()

# (1) Metric anomalies: merge global thresholds, flag breaches (strict > p95, < p5)
mw = metric_window.merge(thresholds, on=['cmdb_id','kpi_name'], how='left')
mw['is_high'] = mw['value'] > mw['p95']
mw['is_low'] = mw['value'] < mw['p5']
mw['is_breach'] = mw['is_high'] | mw['is_low']

# Per (cmdb_id, kpi_name) breach counts within window (for top kpis)
metric_kpi_breaches = mw.groupby(['cmdb_id','kpi_name'])['is_breach'].sum().reset_index(name='breach_count')
# Keep only positive breach counts
metric_kpi_breaches = metric_kpi_breaches[metric_kpi_breaches['breach_count']>0]

# Per cmdb_id: earliest_metric_breach and total metric_breach_count
metric_breach_summary = mw[mw['is_breach']].copy()
if metric_breach_summary.shape[0] == 0:
    metric_by_comp = pd.DataFrame(columns=['cmdb_id','earliest_metric_breach','metric_breach_count','metric_kpis_by_breaches'])
else:
    earliest = metric_breach_summary.groupby('cmdb_id')['_ts_dt'].min().reset_index(name='earliest_metric_breach')
    earliest['earliest_metric_breach'] = earliest['earliest_metric_breach'].dt.tz_convert(tz).apply(lambda x: x.isoformat())
    counts = metric_breach_summary.groupby('cmdb_id')['is_breach'].sum().reset_index(name='metric_breach_count')
    # Top 3 KPIs per cmdb_id by breach_count
    def top_kpis(df):
        top = df.sort_values('breach_count', ascending=False).head(3)
        return "; ".join([f"{r.kpi_name}({int(r.breach_count)})" for r in top.itertuples(index=False)])
    top_kpis = metric_kpi_breaches.groupby('cmdb_id').apply(top_kpis).reset_index(name='metric_kpis_by_breaches')
    metric_by_comp = earliest.merge(counts, on='cmdb_id').merge(top_kpis, on='cmdb_id')

# (2) Trace anomalies: merge thresholds, flag breaches
tw = trace_window.merge(trace_thresholds, on=['cmdb_id','trace_name'], how='left')
tw['is_high'] = tw['value'] > tw['p95']
tw['is_low'] = tw['value'] < tw['p5']
tw['is_breach'] = tw['is_high'] | tw['is_low']

trace_name_breaches = tw.groupby(['cmdb_id','trace_name'])['is_breach'].sum().reset_index(name='breach_count')
trace_name_breaches = trace_name_breaches[trace_name_breaches['breach_count']>0]

trace_breach_summary = tw[tw['is_breach']].copy()
if trace_breach_summary.shape[0] == 0:
    trace_by_comp = pd.DataFrame(columns=['cmdb_id','earliest_trace_breach','trace_breach_count','trace_names_by_breaches'])
else:
    earliest_t = trace_breach_summary.groupby('cmdb_id')['_ts_dt'].min().reset_index(name='earliest_trace_breach')
    earliest_t['earliest_trace_breach'] = earliest_t['earliest_trace_breach'].dt.tz_convert(tz).apply(lambda x: x.isoformat())
    counts_t = trace_breach_summary.groupby('cmdb_id')['is_breach'].sum().reset_index(name='trace_breach_count')
    def top_traces(df):
        top = df.sort_values('breach_count', ascending=False).head(3)
        return "; ".join([f"{r.trace_name}({int(r.breach_count)})" for r in top.itertuples(index=False)])
    top_traces = trace_name_breaches.groupby('cmdb_id').apply(top_traces).reset_index(name='trace_names_by_breaches')
    trace_by_comp = earliest_t.merge(counts_t, on='cmdb_id').merge(top_traces, on='cmdb_id')

# (3) Join metric and trace summaries by cmdb_id; include rows where either count > 0
all_cmdbs = pd.merge(metric_by_comp, trace_by_comp, on='cmdb_id', how='outer')

# Fill missing columns for components with only one side present
for col in ['earliest_metric_breach','metric_breach_count','metric_kpis_by_breaches']:
    if col not in all_cmdbs.columns:
        all_cmdbs[col] = pd.NA
for col in ['earliest_trace_breach','trace_breach_count','trace_names_by_breaches']:
    if col not in all_cmdbs.columns:
        all_cmdbs[col] = pd.NA

all_cmdbs['metric_breach_count'] = all_cmdbs['metric_breach_count'].fillna(0).astype(int)
all_cmdbs['trace_breach_count'] = all_cmdbs['trace_breach_count'].fillna(0).astype(int)

# Filter rows where either metric_breach_count>0 or trace_breach_count>0
all_cmdbs['combined_breach_count'] = all_cmdbs['metric_breach_count'] + all_cmdbs['trace_breach_count']
result = all_cmdbs[(all_cmdbs['combined_breach_count']>0)].copy()

# Order by combined_breach_count desc and limit top 20
result = result.sort_values(by='combined_breach_count', ascending=False).head(20)

# Select and order columns as requested
final_cols = [
    'cmdb_id',
    'earliest_metric_breach','metric_breach_count','metric_kpis_by_breaches',
    'earliest_trace_breach','trace_breach_count','trace_names_by_breaches'
]
# Ensure all columns exist
for c in final_cols:
    if c not in result.columns:
        result[c] = pd.NA

result = result[final_cols].reset_index(drop=True)

# Compact output (top 20)
result

```
Out[6]:
```
```python
# Build a concise plain-English summary of the correlation table stored in `result`
try:
    result
except NameError:
    # If `result` is not present, produce a fallback message
    summary = "Correlation result not found in the session. Please run the correlation analysis first."
else:
    n_rows = result.shape[0]
    top_rows = result.head(8)
    lines = []
    for i, r in top_rows.iterrows():
        lines.append(
            f"{i+1}. {r['cmdb_id']} — metrics: earliest={r['earliest_metric_breach']} count={r['metric_breach_count']} "
            f"top_kpis=[{(r['metric_kpis_by_breaches'][:120] + '...') if pd.notna(r['metric_kpis_by_breaches']) and len(r['metric_kpis_by_breaches'])>120 else r['metric_kpis_by_breaches']}] "
            f"| traces: earliest={r['earliest_trace_breach']} count={r['trace_breach_count']} "
            f"top_traces=[{(r['trace_names_by_breaches'][:120] + '...') if pd.notna(r['trace_names_by_breaches']) and len(r['trace_names_by_breaches'])>120 else r['trace_names_by_breaches']}]"
        )

    summary = (
        f"Correlation summary for incident window (2022-03-20 20:00–20:30 Asia/Shanghai):\n"
        f"- Components with metric or trace breaches: {n_rows} (showing top {min(8, n_rows)} below).\n"
        "- Quick interpretation:\n"
        "  * Frontend components (frontend-2, frontend-1, frontend2-0, frontend-0) appear at the top by combined\n"
        "    breach counts — they have many metric breaches and many trace anomalies starting at or soon after 20:00.\n"
        "  * Checkoutservice and productcatalogservice variants also show substantial metric and trace breach activity\n"
        "    with earliest breaches at ~20:00–20:03.\n"
        "  * Common metric types among top components include container CPU/memory, network transmit/receive, and\n"
        "    mesh/istio metrics (seen in the top-kpis strings). Trace anomalies commonly involve frontend→checkout/cart\n"
        "    interaction traces (see top_traces strings).\n"
        "- Actionable next steps:\n"
        "  1) Start with components listed above and correlate the earliest_metric_breach and earliest_trace_breach\n"
        "     timestamps (shown) to identify whether trace latency/spikes coincide with metric resource anomalies.\n"
        "  2) Inspect the top metric KPIs and trace names for each high-ranked component to pinpoint resource vs\n"
        "     request-path issues (e.g., memory/cpu/fs vs frontend→checkout/cart latency).\n\n"
        "Top components (examples):\n  " + "\n  ".join(lines)
    )

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id     earliest_metric_breach  metric_breach_count                            metric_kpis_by_breaches      earliest_trace_breach  trace_breach_count                            trace_names_by_breaches
0                 frontend-2  2022-03-20T20:00:00+08:00                  222  container.node-5.container_network_transmit_MB...  2022-03-20T20:00:00+08:00                 190  trace.from_root.duration_mean(4); trace.from_r...
1                 frontend-1  2022-03-20T20:02:00+08:00                  173  mesh.source.frontend.jaeger-collector.istio_re...  2022-03-20T20:00:00+08:00                 207  trace.from_root.duration_mean(4); trace.from_r...
2                frontend2-0  2022-03-20T20:02:00+08:00                  230  mesh.source.frontend2.cartservice2.istio_reque...  2022-03-20T20:00:00+08:00                  91  trace.from_root.duration_mean(4); trace.from_r...
3                 frontend-0  2022-03-20T20:02:00+08:00                  129  mesh.source.frontend.cartservice.istio_request...  2022-03-20T20:00:00+08:00                 190  trace.from_root.duration_mean(4); trace.from_r...
4          checkoutservice-1  2022-03-20T20:00:00+08:00                  168  container.node-6.container_cpu_usage_seconds(5...  2022-03-20T20:00:00+08:00                 147  trace.from_frontend-1.duration_mean(4); trace....
5         checkoutservice2-0  2022-03-20T20:02:00+08:00                  171  mesh.source.checkoutservice2.currencyservice2....  2022-03-20T20:00:00+08:00                  83  trace.from_frontend2-0.duration_mean(4); trace...
6    productcatalogservice-2  2022-03-20T20:00:00+08:00                  111  mesh.source.productcatalogservice.jaeger-colle...  2022-03-20T20:00:00+08:00                  90  trace.from_checkoutservice-0.duration_mean(4);...
7          checkoutservice-2  2022-03-20T20:03:00+08:00                   50  container.node-5.container_network_transmit_pa...  2022-03-20T20:00:00+08:00                 142  trace.from_frontend-1.duration_mean(4); trace....
8          checkoutservice-0  2022-03-20T20:02:00+08:00                   47  container.node-6.container_cpu_usage_seconds(3...  2022-03-20T20:00:00+08:00                 142  trace.from_frontend-1.duration_mean(4); trace....
9    productcatalogservice-0  2022-03-20T20:00:00+08:00                  100  mesh.destination.recommendationservice2.produc...  2022-03-20T20:00:00+08:00                  88  trace.from_checkoutservice-0.duration_mean(4);...
10   productcatalogservice-1  2022-03-20T20:03:00+08:00                   97  container.node-6.container_network_receive_MB....  2022-03-20T20:00:00+08:00                  84  trace.from_checkoutservice-0.duration_mean(4);...
11        shippingservice2-0  2022-03-20T20:02:00+08:00                  122  container.node-6.container_cpu_usage_seconds(6...  2022-03-20T20:05:00+08:00                  19  trace.from_checkoutservice2-0.duration_mean(4)...
12   recommendationservice-0  2022-03-20T20:00:00+08:00                   80  container.node-6.container_network_receive_MB....  2022-03-20T20:00:00+08:00                  59  trace.from_frontend-0.duration_mean(4); trace....
13         currencyservice-0  2022-03-20T20:02:00+08:00                   77  container.node-6.container_fs_usage_MB./dev/vd...  2022-03-20T20:00:00+08:00                  56  trace.from_checkoutservice-0.duration_mean(4);...
14  productcatalogservice2-0  2022-03-20T20:02:00+08:00                  106  container.node-6.container_network_receive_MB....  2022-03-20T20:00:00+08:00                  23  trace.from_checkoutservice2-0.duration_mean(4)...
15             cartservice-1  2022-03-20T20:03:00+08:00                   97  mesh.destination.frontend.cartservice.istio_re...  2022-03-20T20:00:00+08:00                  32  trace.self.row_count(4); trace.from_frontend-1...
16   recommendationservice-2  2022-03-20T20:00:00+08:00                   73  container.node-6.container_network_receive_MB....  2022-03-20T20:00:00+08:00                  52  trace.from_frontend-0.duration_mean(4); trace....
17         currencyservice-1  2022-03-20T20:07:00+08:00                   70  container.node-6.container_fs_usage_MB./dev/vd...  2022-03-20T20:00:00+08:00                  53  trace.from_checkoutservice-0.duration_mean(4);...
18         currencyservice-2  2022-03-20T20:04:00+08:00                   66  container.node-6.container_network_receive_MB....  2022-03-20T20:00:00+08:00                  57  trace.from_checkoutservice-0.duration_mean(4);...
19            cartservice2-0  2022-03-20T20:02:00+08:00                   98  mesh.destination.frontend2.cartservice2.istio_...  2022-03-20T20:00:00+08:00                  19  trace.from_checkoutservice2-0.duration_mean(4)...```
```

In [ ]:
# In[7]:

import pandas as pd
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Reuse existing dataframes if present, otherwise load
try:
    log
except NameError:
    log = pd.read_csv('log.csv')

try:
    error_logs
except NameError:
    error_logs = pd.read_csv('error_logs.csv')

# Ensure types and timezone-aware timestamps (Asia/Shanghai)
log['timestamp'] = pd.to_numeric(log['timestamp'], errors='coerce')
log['value'] = pd.to_numeric(log['value'], errors='coerce')
log['_ts_dt'] = pd.to_datetime(log['timestamp'], unit='s', utc=True).dt.tz_convert(tz)

error_logs['timestamp'] = pd.to_numeric(error_logs['timestamp'], errors='coerce') if 'timestamp' in error_logs.columns else pd.Series(dtype='float')
if 'timestamp' in error_logs.columns:
    error_logs['_ts_dt'] = pd.to_datetime(error_logs['timestamp'], unit='s', utc=True).dt.tz_convert(tz)

# Allowed components (nodes, pods, services) - reuse if present else define
try:
    allowed_cmdb
except NameError:
    nodes = [f"node-{i}" for i in range(1,7)]
    pods = [
        "frontend-0","frontend-1","frontend-2","frontend2-0",
        "shippingservice-0","shippingservice-1","shippingservice-2","shippingservice2-0",
        "checkoutservice-0","checkoutservice-1","checkoutservice-2","checkoutservice2-0",
        "currencyservice-0","currencyservice-1","currencyservice-2","currencyservice2-0",
        "adservice-0","adservice-1","adservice-2","adservice2-0",
        "emailservice-0","emailservice-1","emailservice-2","emailservice2-0",
        "cartservice-0","cartservice-1","cartservice-2","cartservice2-0",
        "productcatalogservice-0","productcatalogservice-1","productcatalogservice-2","productcatalogservice2-0",
        "recommendationservice-0","recommendationservice-1","recommendationservice-2","recommendationservice2-0",
        "paymentservice-0","paymentservice-1","paymentservice-2","paymentservice2-0"
    ]
    services = [
        "frontend","shippingservice","checkoutservice","currencyservice","adservice",
        "emailservice","cartservice","productcatalogservice","recommendationservice","paymentservice"
    ]
    allowed_cmdb = set(nodes + pods + services)

# Incident window (inclusive) in Asia/Shanghai
start_dt = tz.localize(pd.to_datetime("2022-03-20 20:00:00"))
end_dt = tz.localize(pd.to_datetime("2022-03-20 20:30:00"))

# Filter log rows to window and allowed components
log_window = log[(log['_ts_dt'] >= start_dt) & (log['_ts_dt'] <= end_dt) & (log['cmdb_id'].isin(allowed_cmdb))].copy()

# 1) Per cmdb_id aggregates for log.error_count
# Total rows in window per cmdb
total_rows = log_window.groupby('cmdb_id').size().reset_index(name='total_rows_in_window')

# Rows where log_name == 'log.error_count'
error_rows = log_window[log_window['log_name'] == 'log.error_count'].copy()
# total_error_count: sum of value (treat NaN as 0)
error_sum = error_rows.groupby('cmdb_id')['value'].sum().reset_index(name='total_error_count')

# count_error_rows: number of rows with log.error_count > 0
error_positive = error_rows[error_rows['value'] > 0].copy()
count_error_rows = error_positive.groupby('cmdb_id').size().reset_index(name='count_error_rows')

# earliest_error_timestamp: first timestamp where log.error_count > 0
if not error_positive.empty:
    earliest_err = error_positive.groupby('cmdb_id')['_ts_dt'].min().reset_index(name='earliest_error_timestamp')
    earliest_err['earliest_error_timestamp'] = earliest_err['earliest_error_timestamp'].dt.tz_convert(tz).apply(lambda x: x.isoformat())
else:
    earliest_err = pd.DataFrame(columns=['cmdb_id','earliest_error_timestamp'])

# Merge metric parts
part1 = total_rows.merge(error_sum, on='cmdb_id', how='left') \
                  .merge(count_error_rows, on='cmdb_id', how='left') \
                  .merge(earliest_err, on='cmdb_id', how='left')

# Fill NaNs for counts with zeros and format
part1['total_error_count'] = part1['total_error_count'].fillna(0)
part1['count_error_rows'] = part1['count_error_rows'].fillna(0).astype(int)
part1['total_rows_in_window'] = part1['total_rows_in_window'].astype(int)

# Order by total_error_count desc and limit top 20
part1 = part1.sort_values('total_error_count', ascending=False).head(20).reset_index(drop=True)

# 2) Raw log rows in-window where log_name == 'log.error_count' and value > 0
raw_error_rows = error_positive.copy()
if not raw_error_rows.empty:
    raw_error_rows['timestamp_readable'] = raw_error_rows['_ts_dt'].apply(lambda x: x.isoformat())
    raw_error_rows = raw_error_rows.sort_values('_ts_dt', ascending=True)
    raw_error_rows = raw_error_rows.loc[:, ['timestamp_readable','cmdb_id','log_name','value']].head(50)
    # rename timestamp column to 'timestamp' for output brevity
    raw_error_rows = raw_error_rows.rename(columns={'timestamp_readable':'timestamp'})
else:
    raw_error_rows = pd.DataFrame(columns=['timestamp','cmdb_id','log_name','value'])

# 3) Inspect error_logs.csv full file
if error_logs.shape[0] == 0:
    error_logs_info = "error_logs.csv is empty"
else:
    # prepare top 20 rows with localized timestamp if available
    if '_ts_dt' in error_logs.columns:
        el = error_logs.copy()
        el['_ts_dt'] = pd.to_datetime(el['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
        el['timestamp_readable'] = el['_ts_dt'].apply(lambda x: x.isoformat())
        error_logs_top20 = el.loc[:, ['timestamp_readable','cmdb_id','message']].head(20).rename(columns={'timestamp_readable':'timestamp'})
    else:
        error_logs_top20 = error_logs.loc[:, ['timestamp','cmdb_id','message']].head(20)

    error_logs_info = error_logs_top20

# Final compact outputs
part1, raw_error_rows, error_logs_info

```
Out[7]:
```
```python
# Compact plain-English summary of the log inspection results (uses previously computed variables)
try:
    part1, raw_error_rows, error_logs_info
except NameError:
    summary = "Log inspection results are not available in the session."
else:
    # Determine top examples from part1
    if part1.shape[0] == 0:
        top_examples = "No log rows for allowed components in the incident window."
    else:
        # show first 6 components as examples
        ex = part1.head(6)
        ex_lines = []
        for r in ex.itertuples(index=False):
            cmdb = r.cmdb_id
            rows = int(r.total_rows_in_window)
            total_err = float(r.total_error_count)
            cnt_err_rows = int(r.count_error_rows)
            ex_lines.append(f"{cmdb}: rows_in_window={rows}, total_error_count={int(total_err)}, count_error_rows={cnt_err_rows}")
        top_examples = "; ".join(ex_lines)

    # Raw error log rows present?
    raw_count = 0 if raw_error_rows.empty else raw_error_rows.shape[0]

    # error_logs.csv presence
    if isinstance(error_logs_info, str) and "empty" in error_logs_info:
        error_logs_msg = "error_logs.csv is empty."
    else:
        # non-empty: report how many top rows available
        error_logs_msg = f"error_logs.csv contains rows (showing top {min(20, error_logs_info.shape[0])})."

    summary = (
        "Log inspection (2022-03-20 20:00:00 to 20:30:00 Asia/Shanghai) — compact summary:\n\n"
        "1) Per-component log.error_count (top 20 by total_error_count):\n"
        "- All returned components have total_error_count = 0 (no summed errors) within the window.\n"
        f"- Example components: {top_examples}\n\n"
        "2) Raw log.error_count entries with value > 0 in-window:\n"
        f"- Count = {raw_count}. (No raw log.error_count > 0 rows found.)\n\n"
        "3) error_logs.csv file:\n"
        f"- {error_logs_msg}\n\n"
        "Conclusion:\n"
        "- There is no evidence of log.error_count or error_logs entries for the allowed components during the\n"
        "  incident window. Investigate metrics/traces and broaden log searches (other log levels/messages or\n"
        "  related components) if runtime errors are still suspected."
    )

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(              cmdb_id  total_rows_in_window  total_error_count  count_error_rows earliest_error_timestamp
0         adservice-0                    60                0.0                 0                      NaN
1         adservice-1                    60                0.0                 0                      NaN
2         adservice-2                    60                0.0                 0                      NaN
3       cartservice-0                    62                0.0                 0                      NaN
4       cartservice-1                    62                0.0                 0                      NaN
5       cartservice-2                    62                0.0                 0                      NaN
6   checkoutservice-0                    60                0.0                 0                      NaN
7   checkoutservice-1                    60                0.0                 0                      NaN
8   checkoutservice-2                    60                0.0                 0                      NaN
9   currencyservice-0                    62                0.0                 0                      NaN
10  currencyservice-1                    62                0.0                 0                      NaN
11  currencyservice-2                    62                0.0                 0                      NaN
12     emailservice-0                    60                0.0                 0                      NaN
13     emailservice-1                    60                0.0                 0                      NaN
14     emailservice-2                    60                0.0                 0                      NaN
15         frontend-0                    62                0.0                 0                      NaN
16         frontend-1                    62                0.0                 0                      NaN
17         frontend-2                    62                0.0                 0                      NaN
18   paymentservice-0                    60                0.0                 0                      NaN
19   paymentservice-1                    54                0.0                 0                      NaN, Empty DataFrame
Columns: [timestamp, cmdb_id, log_name, value]
Index: [], 'error_logs.csv is empty')```
```

In [ ]:
# In[8]:

import pandas as pd
import pytz
import numpy as np

tz = pytz.timezone('Asia/Shanghai')

# Reuse previously computed variables where possible: `result`, `trace`, `trace_thresholds`
# Defensive checks and minimal recompute if missing
try:
    result
except NameError:
    raise RuntimeError("Required variable `result` (correlation table) not found in session.")

try:
    trace
except NameError:
    trace = pd.read_csv('trace.csv')

# Ensure trace timestamps and types
trace['timestamp'] = pd.to_numeric(trace['timestamp'], errors='coerce')
trace['value'] = pd.to_numeric(trace['value'], errors='coerce')
trace['_ts_dt'] = pd.to_datetime(trace['timestamp'], unit='s', utc=True).dt.tz_convert(tz)

# Allowed components set (reuse if present)
try:
    allowed_cmdb
except NameError:
    nodes = [f"node-{i}" for i in range(1,7)]
    pods = [
        "frontend-0","frontend-1","frontend-2","frontend2-0",
        "shippingservice-0","shippingservice-1","shippingservice-2","shippingservice2-0",
        "checkoutservice-0","checkoutservice-1","checkoutservice-2","checkoutservice2-0",
        "currencyservice-0","currencyservice-1","currencyservice-2","currencyservice2-0",
        "adservice-0","adservice-1","adservice-2","adservice2-0",
        "emailservice-0","emailservice-1","emailservice-2","emailservice2-0",
        "cartservice-0","cartservice-1","cartservice-2","cartservice2-0",
        "productcatalogservice-0","productcatalogservice-1","productcatalogservice-2","productcatalogservice2-0",
        "recommendationservice-0","recommendationservice-1","recommendationservice-2","recommendationservice2-0",
        "paymentservice-0","paymentservice-1","paymentservice-2","paymentservice2-0"
    ]
    services = [
        "frontend","shippingservice","checkoutservice","currencyservice","adservice",
        "emailservice","cartservice","productcatalogservice","recommendationservice","paymentservice"
    ]
    allowed_cmdb = set(nodes + pods + services)

# Ensure global trace thresholds exist (computed from full trace series)
try:
    trace_thresholds
except NameError:
    trace_thresholds = trace.groupby(['cmdb_id','trace_name'])['value'].agg(
        p95=lambda x: x.quantile(0.95), p5=lambda x: x.quantile(0.05)
    ).reset_index()

# Incident window (inclusive)
start_dt = tz.localize(pd.to_datetime("2022-03-20 20:00:00"))
end_dt = tz.localize(pd.to_datetime("2022-03-20 20:30:00"))

# Windowed trace rows restricted to allowed components
trace_window = trace[(trace['_ts_dt'] >= start_dt) & (trace['_ts_dt'] <= end_dt) & (trace['cmdb_id'].isin(allowed_cmdb))].copy()

# Merge global thresholds into windowed trace rows and flag breaches (do NOT recompute thresholds here)
tw = trace_window.merge(trace_thresholds, on=['cmdb_id','trace_name'], how='left')
tw['is_breach'] = (tw['value'] > tw['p95']) | (tw['value'] < tw['p5'])

# Keep only breached trace rows for matching
tw_breach = tw[tw['is_breach']].copy()
# Normalize trace_name to string
tw_breach['trace_name'] = tw_breach['trace_name'].astype(str)

# Prepare list of cmdbs with metric breaches from `result`
cmdbs_with_metric_breaches = result[result['metric_breach_count'] > 0]['cmdb_id'].unique().tolist()

# For each cmdb_id compute target/source trace breach aggregates
rows = []
for cmdb in cmdbs_with_metric_breaches:
    # Filter trace breaches where trace_name contains 'to_<cmdb>' (as target)
    target_pattern = f"to_{cmdb}"
    source_pattern = f"from_{cmdb}"
    as_target = tw_breach[tw_breach['trace_name'].str.contains(target_pattern, na=False)]
    as_source = tw_breach[tw_breach['trace_name'].str.contains(source_pattern, na=False)]
    # Earliest timestamps (ISO) or NaT
    if not as_target.empty:
        earliest_target = as_target['_ts_dt'].min().tz_convert(tz).isoformat()
        count_target = int(as_target.shape[0])
    else:
        earliest_target = pd.NaT
        count_target = 0
    if not as_source.empty:
        earliest_source = as_source['_ts_dt'].min().tz_convert(tz).isoformat()
        count_source = int(as_source.shape[0])
    else:
        earliest_source = pd.NaT
        count_source = 0
    # Get earliest_metric_breach and metric_breach_count from result
    rr = result[result['cmdb_id'] == cmdb].iloc[0]
    earliest_metric_breach = rr['earliest_metric_breach']
    metric_breach_count = int(rr['metric_breach_count'])
    # role derivation
    if count_target > count_source:
        role = 'mostly_target'
    elif count_source > count_target:
        role = 'mostly_source'
    else:
        role = 'mixed'
    combined = count_target + count_source
    rows.append({
        'cmdb_id': cmdb,
        'earliest_metric_breach': earliest_metric_breach,
        'metric_breach_count': metric_breach_count,
        'earliest_trace_breach_as_target': earliest_target,
        'count_trace_breach_as_target': count_target,
        'earliest_trace_breach_as_source': earliest_source,
        'count_trace_breach_as_source': count_source,
        'role': role,
        'combined_trace_breach_count': combined
    })

localization_df = pd.DataFrame(rows)

# Order by combined_trace_breach_count desc and limit top 20
localization_df = localization_df.sort_values(by='combined_trace_breach_count', ascending=False).head(20).reset_index(drop=True)

# Final compact output
localization_df

```
Out[8]:
```
```python
# Build a compact plain-English summary from the previously computed `localization_df`
try:
    localization_df
except NameError:
    summary = "Localization results not found in session."
else:
    n = localization_df.shape[0]
    topn = localization_df.head(8)
    lines = []
    for i, r in topn.iterrows():
        lines.append(
            f"{i+1}. {r['cmdb_id']}: earliest_metric_breach={r['earliest_metric_breach']}, "
            f"metric_breach_count={int(r['metric_breach_count'])}, "
            f"trace_as_target={int(r['count_trace_breach_as_target'])} (earliest={r['earliest_trace_breach_as_target']}), "
            f"trace_as_source={int(r['count_trace_breach_as_source'])} (earliest={r['earliest_trace_breach_as_source']}), "
            f"role={r['role']}"
        )

    # Identify top candidate components acting as downstream targets (role == 'mostly_target') from the table
    targets = localization_df[localization_df['role']=='mostly_target'].head(5)['cmdb_id'].tolist()
    sources = localization_df[localization_df['role']=='mostly_source'].head(5)['cmdb_id'].tolist()

    summary = (
        f"Downstream-localization summary (components with metric breaches):\n"
        f"- Components analyzed: {n} (showing top {min(8,n)} by combined trace breaches).\n\n"
        f"- Top examples (earliest_metric_breach, metric_breach_count, trace_as_target/count, trace_as_source/count, role):\n  "
        + "\n  ".join(lines) + "\n\n"
        f"- Observations:\n"
        f"  * Many frontend and checkout-related components appear as strong sources of anomalous traces\n"
        f"    (roles labeled 'mostly_source'), with earliest trace anomalies around 20:00 and metric breaches\n"
        f"    shortly thereafter.\n"
        f"  * Several productcatalogservice and currencyservice components appear as 'mostly_target' — they\n"
        f"    receive a high number of breached traces (trace_as_target) shortly after their metric breaches;\n"
        f"    these are high-priority downstream candidates for root-cause investigation.\n\n"
        f"- Quick candidate lists:\n"
        f"  * Mostly-target examples (prioritize): {', '.join(targets) if targets else 'none'}\n"
        f"  * Mostly-source examples: {', '.join(sources) if sources else 'none'}\n\n"
        f"- Recommendation: start troubleshooting with the 'mostly_target' components (above) because they are\n"
        f"  frequently observed as downstream targets of anomalous traces soon after their metric breaches — this\n"
        f"  temporal relationship makes them strong root-cause candidates to investigate (inspect logs, GC,\n"
        f"  memory/FS metrics, and request handling around the earliest_metric_breach timestamps).\n"
    )

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id     earliest_metric_breach  metric_breach_count earliest_trace_breach_as_target  count_trace_breach_as_target earliest_trace_breach_as_source  count_trace_breach_as_source           role  combined_trace_breach_count
0                 frontend-1  2022-03-20T20:02:00+08:00                  173                             NaT                             0       2022-03-20T20:00:00+08:00                           183  mostly_source                          183
1                 frontend-2  2022-03-20T20:00:00+08:00                  222                             NaT                             0       2022-03-20T20:00:00+08:00                           167  mostly_source                          167
2                 frontend-0  2022-03-20T20:02:00+08:00                  129                             NaT                             0       2022-03-20T20:00:00+08:00                           167  mostly_source                          167
3          checkoutservice-1  2022-03-20T20:00:00+08:00                  168       2022-03-20T20:03:00+08:00                            16       2022-03-20T20:00:00+08:00                           113  mostly_source                          129
4          checkoutservice-2  2022-03-20T20:03:00+08:00                   50       2022-03-20T20:05:00+08:00                            16       2022-03-20T20:00:00+08:00                           108  mostly_source                          124
5          checkoutservice-0  2022-03-20T20:02:00+08:00                   47       2022-03-20T20:00:00+08:00                            16       2022-03-20T20:00:00+08:00                           104  mostly_source                          120
6    productcatalogservice-2  2022-03-20T20:00:00+08:00                  111       2022-03-20T20:00:00+08:00                            90                             NaT                             0  mostly_target                           90
7    productcatalogservice-0  2022-03-20T20:00:00+08:00                  100       2022-03-20T20:00:00+08:00                            88                             NaT                             0  mostly_target                           88
8    productcatalogservice-1  2022-03-20T20:03:00+08:00                   97       2022-03-20T20:00:00+08:00                            84                             NaT                             0  mostly_target                           84
9                frontend2-0  2022-03-20T20:02:00+08:00                  230                             NaT                             0       2022-03-20T20:00:00+08:00                            67  mostly_source                           67
10        checkoutservice2-0  2022-03-20T20:02:00+08:00                  171       2022-03-20T20:07:00+08:00                             9       2022-03-20T20:00:00+08:00                            52  mostly_source                           61
11         currencyservice-2  2022-03-20T20:04:00+08:00                   66       2022-03-20T20:00:00+08:00                            57                             NaT                             0  mostly_target                           57
12         currencyservice-0  2022-03-20T20:02:00+08:00                   77       2022-03-20T20:00:00+08:00                            56                             NaT                             0  mostly_target                           56
13         currencyservice-1  2022-03-20T20:07:00+08:00                   70       2022-03-20T20:00:00+08:00                            53                             NaT                             0  mostly_target                           53
14   recommendationservice-0  2022-03-20T20:00:00+08:00                   80       2022-03-20T20:00:00+08:00                            33       2022-03-20T20:00:00+08:00                            16  mostly_target                           49
15   recommendationservice-2  2022-03-20T20:00:00+08:00                   73       2022-03-20T20:00:00+08:00                            28       2022-03-20T20:00:00+08:00                            14  mostly_target                           42
16             cartservice-1  2022-03-20T20:03:00+08:00                   97       2022-03-20T20:00:00+08:00                            26                             NaT                             0  mostly_target                           26
17  productcatalogservice2-0  2022-03-20T20:02:00+08:00                  106       2022-03-20T20:00:00+08:00                            23                             NaT                             0  mostly_target                           23
18        shippingservice2-0  2022-03-20T20:02:00+08:00                  122       2022-03-20T20:05:00+08:00                            19                             NaT                             0  mostly_target                           19
19            cartservice2-0  2022-03-20T20:02:00+08:00                   98       2022-03-20T20:00:00+08:00                            13                             NaT                             0  mostly_target                           13```
```

In [ ]:
# In[9]:

import pandas as pd
import pytz
import numpy as np

tz = pytz.timezone('Asia/Shanghai')

# Reuse existing `metric` if present, otherwise load
try:
    metric
except NameError:
    metric = pd.read_csv('metric.csv')

# Ensure types and timezone-aware timestamp column (Asia/Shanghai)
metric['timestamp'] = pd.to_numeric(metric['timestamp'], errors='coerce')
metric['value'] = pd.to_numeric(metric['value'], errors='coerce')
metric['_ts_dt'] = pd.to_datetime(metric['timestamp'], unit='s', utc=True).dt.tz_convert(tz)

# Target components (priority list)
targets = [
    "productcatalogservice-2",
    "productcatalogservice-0",
    "productcatalogservice-1",
    "productcatalogservice2-0",
    "currencyservice-0",
    "currencyservice-1",
    "currencyservice-2"
]

# Filter metric to only the target components but compute aggregations using the full series for those components
metric_targets = metric[metric['cmdb_id'].isin(targets)].copy()

# If no data for targets, return empty DataFrame with schema
if metric_targets.empty:
    kpi_inventory = pd.DataFrame(columns=[
        'cmdb_id','kpi_name','count','min_timestamp','max_timestamp','p95','p5','mean'
    ])
else:
    # Group by (cmdb_id, kpi_name) and compute global stats using the full available series (for those components)
    g = metric_targets.groupby(['cmdb_id','kpi_name'])
    agg = g.agg(
        count = ('value','count'),
        p95 = ('value', lambda x: x.quantile(0.95)),
        p5  = ('value', lambda x: x.quantile(0.05)),
        mean = ('value','mean'),
        min_ts = ('_ts_dt','min'),
        max_ts = ('_ts_dt','max')
    ).reset_index()

    # Format timestamps as Asia/Shanghai ISO strings
    agg['min_timestamp'] = agg['min_ts'].apply(lambda x: x.tz_convert(tz).isoformat() if pd.notnull(x) else pd.NaT)
    agg['max_timestamp'] = agg['max_ts'].apply(lambda x: x.tz_convert(tz).isoformat() if pd.notnull(x) else pd.NaT)

    # Select and order columns as requested
    agg = agg[['cmdb_id','kpi_name','count','min_timestamp','max_timestamp','p95','p5','mean']]

    # For each component, keep top 20 KPI groups by count, then limit overall to 100 rows
    agg = agg.sort_values(['cmdb_id','count'], ascending=[True, False])
    kpi_inventory = agg.groupby('cmdb_id').head(20).reset_index(drop=True).head(100)

# Final compact output
kpi_inventory

```
Out[9]:
```
```python
# Build a concise plain-English summary from the computed `kpi_inventory`
import pandas as pd
import pytz

tz = pytz.timezone('Asia/Shanghai')

try:
    kpi_inventory
except NameError:
    raise RuntimeError("kpi_inventory not found in session. Please run the KPI inventory step first.")

# Priority components (as requested)
targets = [
    "productcatalogservice-2",
    "productcatalogservice-0",
    "productcatalogservice-1",
    "productcatalogservice2-0",
    "currencyservice-0",
    "currencyservice-1",
    "currencyservice-2"
]

summary_lines = []
summary_lines.append("KPI inventory summary for priority 'mostly_target' components (full series, timestamps in Asia/Shanghai):")
summary_lines.append("Overall note: most KPI groups have count=61 and cover 2022-03-20T19:30:00+08:00 to 2022-03-20T20:30:00+08:00.")

for comp in targets:
    comp_df = kpi_inventory[kpi_inventory['cmdb_id'] == comp].copy()
    if comp_df.empty:
        summary_lines.append(f"- {comp}: no KPIs found in inventory.")
        continue
    total_kpis = int(comp_df.shape[0])
    # Count degenerate KPIs where both p95 and p5 are zero (or both NaN)
    degenerate_mask = (comp_df['p95'].fillna(0) == 0) & (comp_df['p5'].fillna(0) == 0)
    degenerate_count = int(degenerate_mask.sum())
    nondeg = comp_df[~degenerate_mask].sort_values('count', ascending=False)
    # Select up to 5 representative non-degenerate KPIs; if none, pick top 5 by count
    if not nondeg.empty:
        top_kpis = nondeg.head(5)
    else:
        top_kpis = comp_df.sort_values('count', ascending=False).head(5)
    # Format KPI examples: kpi_name(p95,p5,mean,count)
    examples = []
    for r in top_kpis.itertuples(index=False):
        examples.append(f"{r.kpi_name} p95={r.p95:.6g} p5={r.p5:.6g} mean={r.mean:.6g} count={int(r.count)}")
    summary_lines.append(
        f"- {comp}: {total_kpis} KPI groups (degenerate p95/p5 both 0 for {degenerate_count}). "
        f"Representative KPIs: " + "; ".join(examples)
    )

summary_lines.append("")
summary_lines.append("Actionable guidance:")
summary_lines.append("- Prioritize KPIs with non-zero p95/p5 and meaningful means (listed above).")
summary_lines.append("- If many KPIs are degenerate (p95==p5==0), check instrumentation or choose non-degenerate KPIs such as:")
summary_lines.append("  * container_memory_* (memory usage, working set, rss), container_fs_usage_MB, container_cpu_* and")
summary_lines.append("  * container_cpu_cfs_throttled_* or container_cpu_usage_seconds for CPU throttling/usage signals.")
summary_lines.append("- Use these prioritized KPIs for time-series fault detection in the incident window next.")

summary = "\n".join(summary_lines)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id                                           kpi_name  count              min_timestamp              max_timestamp        p95      p5        mean
0         currencyservice-0         container.node-6.container_cpu_cfs_periods     61  2022-03-20T19:30:00+08:00  2022-03-20T20:30:00+08:00  198.00000  136.50  165.631148
1         currencyservice-0  container.node-6.container_cpu_cfs_throttled_p...     61  2022-03-20T19:30:00+08:00  2022-03-20T20:30:00+08:00    0.50000    0.00    0.040984
2         currencyservice-0  container.node-6.container_cpu_cfs_throttled_s...     61  2022-03-20T19:30:00+08:00  2022-03-20T20:30:00+08:00    0.00394    0.00    0.000398
3         currencyservice-0    container.node-6.container_cpu_load_average_10s     61  2022-03-20T19:30:00+08:00  2022-03-20T20:30:00+08:00    0.00000    0.00    0.000000
4         currencyservice-0      container.node-6.container_cpu_system_seconds     61  2022-03-20T19:30:00+08:00  2022-03-20T20:30:00+08:00    0.04000    0.02    0.028525
..                      ...                                                ...    ...                        ...                        ...        ...     ...         ...
95  productcatalogservice-1       container.node-6.container_fs_reads./dev/vda     61  2022-03-20T19:30:00+08:00  2022-03-20T20:30:00+08:00    0.00000    0.00    0.000000
96  productcatalogservice-1      container.node-6.container_fs_reads./dev/vda1     61  2022-03-20T19:30:00+08:00  2022-03-20T20:30:00+08:00    0.00000    0.00    0.000000
97  productcatalogservice-1    container.node-6.container_fs_reads_MB./dev/vda     61  2022-03-20T19:30:00+08:00  2022-03-20T20:30:00+08:00    0.00000    0.00    0.000000
98  productcatalogservice-1  container.node-6.container_fs_reads_merged./de...     61  2022-03-20T19:30:00+08:00  2022-03-20T20:30:00+08:00    0.00000    0.00    0.000000
99  productcatalogservice-1  container.node-6.container_fs_sector_reads./de...     61  2022-03-20T19:30:00+08:00  2022-03-20T20:30:00+08:00    0.00000    0.00    0.000000

[100 rows x 8 columns]

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[10]:

import pandas as pd
import numpy as np
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Reuse existing metric DataFrame; if missing, load it
try:
    metric
except NameError:
    metric = pd.read_csv('metric.csv')

# Ensure numeric types and timezone-aware timestamp column in Asia/Shanghai
metric['timestamp'] = pd.to_numeric(metric['timestamp'], errors='coerce')
metric['value'] = pd.to_numeric(metric['value'], errors='coerce')
metric['_ts_dt'] = pd.to_datetime(metric['timestamp'], unit='s', utc=True).dt.tz_convert(tz)

# Priority components to analyze
targets = [
    "productcatalogservice-2",
    "productcatalogservice-0",
    "productcatalogservice-1",
    "productcatalogservice2-0",
    "currencyservice-0",
    "currencyservice-1",
    "currencyservice-2"
]

# (1) Ensure global thresholds (P95/P5) are computed from full series (do NOT compute after filtering)
try:
    thresholds
except NameError:
    thresholds = metric.groupby(['cmdb_id','kpi_name'])['value'].agg(
        p95=lambda x: x.quantile(0.95), p5=lambda x: x.quantile(0.05)
    ).reset_index()

# Incident window (inclusive) in Asia/Shanghai
start_dt = tz.localize(pd.to_datetime("2022-03-20 20:00:00"))
end_dt = tz.localize(pd.to_datetime("2022-03-20 20:30:00"))

# (2) Filter metric to targets and window
mw = metric[(metric['cmdb_id'].isin(targets)) & (metric['_ts_dt'] >= start_dt) & (metric['_ts_dt'] <= end_dt)].copy()

# Merge global thresholds into the windowed rows
mw = mw.merge(thresholds, on=['cmdb_id','kpi_name'], how='left')

# Prepare container for runs
runs = []

# Process each (cmdb_id, kpi_name)
grouped = mw.groupby(['cmdb_id','kpi_name'])
for (cmdb_id, kpi_name), df in grouped:
    if df.empty:
        continue
    # Determine breach_type per row
    df = df.sort_values('timestamp').reset_index(drop=True)
    df['breach_type'] = np.where(df['value'] > df['p95'], 'high', np.where(df['value'] < df['p5'], 'low', None))
    df_b = df[df['breach_type'].notna()].copy()
    if df_b.empty:
        continue
    # Compute diff seconds between consecutive breached rows (based on full series timestamps)
    df_b['diff_sec'] = df_b['timestamp'].diff().fillna(0).astype(int)
    # Start new run when diff != 60 or breach_type changes
    df_b['new_run'] = (df_b['diff_sec'] != 60) | (df_b['breach_type'] != df_b['breach_type'].shift())
    df_b['run_id'] = df_b['new_run'].cumsum()
    # Aggregate runs
    agg = df_b.groupby('run_id').agg(
        cmdb_id = ('cmdb_id','first'),
        kpi_name = ('kpi_name','first'),
        breach_type = ('breach_type','first'),
        start_ts_num = ('timestamp','min'),
        end_ts_num = ('timestamp','max'),
        length_minutes = ('timestamp','count'),
        max_value_in_run = ('value','max'),
        min_value_in_run = ('value','min'),
        p95 = ('p95','first'),
        p5 = ('p5','first')
    ).reset_index(drop=True)
    # Keep only runs with length >= 2
    agg = agg[agg['length_minutes'] >= 2].copy()
    if agg.empty:
        continue
    # Convert numeric timestamps to ISO strings in Asia/Shanghai
    agg['start_timestamp'] = pd.to_datetime(agg['start_ts_num'], unit='s', utc=True).dt.tz_convert(tz).apply(lambda x: x.isoformat())
    agg['end_timestamp'] = pd.to_datetime(agg['end_ts_num'], unit='s', utc=True).dt.tz_convert(tz).apply(lambda x: x.isoformat())
    # Compute severity_relative per breach_type
    denom_p95 = agg['p95'].abs().clip(lower=1e-9)
    denom_p5  = agg['p5'].abs().clip(lower=1e-9)
    agg['severity_relative'] = 0.0
    # For high runs
    high_mask = agg['breach_type'] == 'high'
    agg.loc[high_mask, 'severity_relative'] = (agg.loc[high_mask, 'max_value_in_run'] - agg.loc[high_mask, 'p95']) / denom_p95[high_mask]
    # For low runs
    low_mask = agg['breach_type'] == 'low'
    agg.loc[low_mask, 'severity_relative'] = (agg.loc[low_mask, 'p5'] - agg.loc[low_mask, 'min_value_in_run']) / denom_p5[low_mask]
    # Collect relevant columns
    agg = agg[[
        'cmdb_id','kpi_name','breach_type','start_timestamp','end_timestamp','length_minutes',
        'max_value_in_run','min_value_in_run','p95','p5','severity_relative'
    ]]
    runs.append(agg)

# Combine runs
if runs:
    runs_df = pd.concat(runs, ignore_index=True)
    # Sort by severity_relative desc and limit to top 20
    runs_top20 = runs_df.sort_values('severity_relative', ascending=False).head(20).reset_index(drop=True)
else:
    runs_top20 = pd.DataFrame(columns=[
        'cmdb_id','kpi_name','breach_type','start_timestamp','end_timestamp','length_minutes',
        'max_value_in_run','min_value_in_run','p95','p5','severity_relative'
    ])

# Final compact output (top 20 runs)
runs_top20

```
Out[10]:
```
```python
# Compose a compact plain-English summary from the previously computed `runs_top20`
try:
    runs_top20
except NameError:
    summary = "No run-level fault detections available in the session."
else:
    if runs_top20.empty:
        summary = "No consecutive (>=2 min) metric breach runs found for the target components in the incident window."
    else:
        # Prepare top 5 example lines
        lines = []
        for i, r in runs_top20.head(5).iterrows():
            lines.append(
                f"{i+1}. {r['cmdb_id']} | {r['kpi_name']} | {r['breach_type']} | "
                f"{r['start_timestamp']} → {r['end_timestamp']} ({int(r['length_minutes'])} min) | "
                f"max={r['max_value_in_run']:.6g} min={r['min_value_in_run']:.6g} | "
                f"p95={r['p95']:.6g} p5={r['p5']:.6g} | severity={r['severity_relative']:.6g}"
            )
        # High-level observations
        obs = [
            "Key findings:",
            "- Highest-severity consecutive breaches (top two):",
            "  * currencyservice-0: container_network_receive_MB (high) at 2022-03-20T20:10–20:11, severity ~2.26",
            "  * currencyservice-2: container_cpu_cfs_throttled_seconds (high) at 2022-03-20T20:05–20:06, severity ~2.14",
            "- Multiple 2-minute consecutive runs were detected across productcatalogservice-* and productcatalogservice2-0",
            "  involving mesh destination/source metrics and some container CPU/network KPIs (mostly length = 2 minutes).",
            "- Many runs have modest severity (<0.25) after the top two; these still indicate repeated anomalies worth checking.",
            "",
            "Practical next steps:",
            "- Immediately investigate currencyservice (network receive spike and CPU throttling) at ~20:05–20:11 (top priority).",
            "- Inspect productcatalogservice instances for mesh-related anomalies and container CPU/network around 20:00–20:03 and 20:21–20:26.",
            "- Correlate these run start times with trace anomalies and logs to confirm causal flow.",
            "- Use KPIs identified earlier (memory, fs_usage_MB, cpu throttling, container_network_*) for time-series troubleshooting."
        ]
        summary = "Top consecutive breach runs (showing top 5 examples):\n\n  " + "\n  ".join(lines) + "\n\n" + "\n".join(obs)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id                                           kpi_name breach_type            start_timestamp              end_timestamp  length_minutes  max_value_in_run  min_value_in_run           p95            p5  severity_relative
0          currencyservice-0  container.node-6.container_network_receive_MB....        high  2022-03-20T20:10:00+08:00  2022-03-20T20:11:00+08:00               2      3.838353e-01      3.731322e-01  1.175704e-01  7.550764e-02           2.264727
1          currencyservice-2  container.node-6.container_cpu_cfs_throttled_s...        high  2022-03-20T20:05:00+08:00  2022-03-20T20:06:00+08:00               2      2.169560e-02      2.169560e-02  6.909102e-03  0.000000e+00           2.140148
2    productcatalogservice-0  mesh.destination.checkoutservice.productcatalo...         low  2022-03-20T20:25:00+08:00  2022-03-20T20:26:00+08:00               2      1.312500e-02      1.060000e-02  3.180000e-02  1.362500e-02           0.222018
3    productcatalogservice-2  mesh.destination.recommendationservice2.produc...        high  2022-03-20T20:00:00+08:00  2022-03-20T20:01:00+08:00               2      3.100000e-03      3.100000e-03  2.625000e-03  0.000000e+00           0.180952
4          currencyservice-2  mesh.destination.frontend.currencyservice.isti...         low  2022-03-20T20:21:00+08:00  2022-03-20T20:22:00+08:00               2      9.817500e-02      9.082500e-02  1.375500e-01  9.817500e-02           0.074866
5    productcatalogservice-2  mesh.source.productcatalogservice.jaeger-colle...        high  2022-03-20T20:02:00+08:00  2022-03-20T20:03:00+08:00               2      8.400000e-03      8.350000e-03  7.875000e-03  1.575000e-03           0.066667
6    productcatalogservice-1  container.node-6.container_network_receive_MB....         low  2022-03-20T20:21:00+08:00  2022-03-20T20:22:00+08:00               2      5.586696e-01      5.333972e-01  8.176041e-01  5.672135e-01           0.059618
7    productcatalogservice-1  container.node-6.container_network_transmit_pa...         low  2022-03-20T20:21:00+08:00  2022-03-20T20:22:00+08:00               2      3.306500e+03      3.149500e+03  4.809000e+03  3.347500e+03           0.059149
8    productcatalogservice-0  mesh.source.productcatalogservice.jaeger-colle...        high  2022-03-20T20:03:00+08:00  2022-03-20T20:04:00+08:00               2      9.450000e-03      9.325000e-03  8.925000e-03  2.100000e-03           0.058824
9    productcatalogservice-1  container.node-6.container_network_receive_pac...         low  2022-03-20T20:21:00+08:00  2022-03-20T20:22:00+08:00               2      3.392500e+03      3.248000e+03  4.889000e+03  3.434000e+03           0.054164
10   productcatalogservice-0  mesh.destination.recommendationservice2.produc...         low  2022-03-20T20:29:00+08:00  2022-03-20T20:30:00+08:00               2      4.917000e+04      4.730750e+04  5.513000e+04  4.991500e+04           0.052239
11   productcatalogservice-0  mesh.destination.recommendationservice2.produc...         low  2022-03-20T20:29:00+08:00  2022-03-20T20:30:00+08:00               2      1.815000e+05      1.746250e+05  2.035000e+05  1.842500e+05           0.052239
12   productcatalogservice-0        container.node-6.container_cpu_user_seconds         low  2022-03-20T20:21:00+08:00  2022-03-20T20:22:00+08:00               2      2.000000e-01      1.950000e-01  3.200000e-01  2.050000e-01           0.048780
13   productcatalogservice-2  mesh.destination.recommendationservice.product...         low  2022-03-20T20:23:00+08:00  2022-03-20T20:24:00+08:00               2      3.200000e+01      3.100000e+01  4.400000e+01  3.250000e+01           0.046154
14   productcatalogservice-2  mesh.destination.recommendationservice.product...         low  2022-03-20T20:23:00+08:00  2022-03-20T20:24:00+08:00               2      3.200000e+01      3.100000e+01  4.400000e+01  3.250000e+01           0.046154
15   productcatalogservice-2  mesh.destination.recommendationservice.product...         low  2022-03-20T20:23:00+08:00  2022-03-20T20:24:00+08:00               2      3.200000e+01      3.100000e+01  4.400000e+01  3.250000e+01           0.046154
16  productcatalogservice2-0  mesh.source.productcatalogservice2.jaeger-coll...         low  2022-03-20T20:29:00+08:00  2022-03-20T20:30:00+08:00               2      1.356725e+06      1.330750e+06  1.511450e+06  1.385500e+06           0.039516
17  productcatalogservice2-0  mesh.source.productcatalogservice2.basic-tidb....         low  2022-03-20T20:29:00+08:00  2022-03-20T20:30:00+08:00               2      9.763610e+05      9.502615e+05  1.095318e+06  9.882570e+05           0.038447
18  productcatalogservice2-0  mesh.source.productcatalogservice2.basic-tidb....         low  2022-03-20T20:29:00+08:00  2022-03-20T20:30:00+08:00               2      1.256500e+03      1.222500e+03  1.409500e+03  1.271000e+03           0.038159
19  productcatalogservice2-0  mesh.source.productcatalogservice2.basic-tidb....         low  2022-03-20T20:29:00+08:00  2022-03-20T20:30:00+08:00               2      3.744370e+05      3.643050e+05  4.200310e+05  3.787580e+05           0.038159```
```

In [ ]:
# In[11]:

import pandas as pd
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Reuse session variables: runs_top20, trace, trace_thresholds
try:
    runs_top20
except NameError:
    # No runs to process
    runs_top20 = pd.DataFrame(columns=[
        'cmdb_id','kpi_name','breach_type','start_timestamp','end_timestamp','length_minutes',
        'max_value_in_run','min_value_in_run','p95','p5','severity_relative'
    ])

try:
    trace
except NameError:
    trace = pd.read_csv('trace.csv')

# Ensure trace timestamps and numeric value
trace['timestamp'] = pd.to_numeric(trace['timestamp'], errors='coerce')
trace['value'] = pd.to_numeric(trace['value'], errors='coerce')
trace['_ts_dt'] = pd.to_datetime(trace['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
trace['trace_name'] = trace['trace_name'].astype(str)

# Ensure global trace thresholds exist (computed from full series)
try:
    trace_thresholds
except NameError:
    trace_thresholds = trace.groupby(['cmdb_id','trace_name'])['value'].agg(
        p95=lambda x: x.quantile(0.95), p5=lambda x: x.quantile(0.05)
    ).reset_index()

# Merge thresholds into trace for breach detection
trace_merged = trace.merge(trace_thresholds, on=['cmdb_id','trace_name'], how='left')
trace_merged['is_breach'] = (trace_merged['value'] > trace_merged['p95']) | (trace_merged['value'] < trace_merged['p5'])

# Limit runs to top 20 by available severity (runs_top20 is assumed sorted by severity)
runs_to_check = runs_top20.head(20).copy()

out_rows = []
for r in runs_to_check.itertuples(index=False):
    cmdb = r.cmdb_id
    kpi = r.kpi_name
    run_start_str = r.start_timestamp
    run_end_str = r.end_timestamp
    length_minutes = int(r.length_minutes) if pd.notnull(r.length_minutes) else None
    severity = float(r.severity_relative) if pd.notnull(r.severity_relative) else None

    # Parse run_start timestamp to tz-aware datetime
    try:
        run_start_dt = pd.to_datetime(run_start_str).tz_convert(tz)
    except Exception:
        run_start_dt = pd.to_datetime(run_start_str, utc=True).dt.tz_convert(tz)

    window_end = run_start_dt + pd.Timedelta(minutes=5)

    # Build match patterns for target/destination: to_<cmdb> OR mesh.destination.<cmdb>
    pat1 = f"to_{cmdb}"
    pat2 = f"mesh.destination.{cmdb}"

    # Filter trace_merged for breaches where trace_name contains patterns and timestamp within [run_start, run_start+5min]
    mask_time = (trace_merged['_ts_dt'] >= run_start_dt) & (trace_merged['_ts_dt'] <= window_end)
    mask_name = trace_merged['trace_name'].str.contains(pat1, na=False) | trace_merged['trace_name'].str.contains(pat2, na=False)
    mask_breach = trace_merged['is_breach'] == True

    candidates = trace_merged[mask_time & mask_name & mask_breach].copy()
    candidates = candidates.sort_values('_ts_dt', ascending=True)

    if not candidates.empty:
        first = candidates.iloc[0]
        earliest_trace_after_run_timestamp = first['_ts_dt'].isoformat()
        earliest_trace_name = first['trace_name']
        earliest_trace_value = first['value']
        earliest_trace_p95 = first['p95']
        earliest_trace_p5 = first['p5']
    else:
        earliest_trace_after_run_timestamp = pd.NaT
        earliest_trace_name = pd.NA
        earliest_trace_value = pd.NA
        earliest_trace_p95 = pd.NA
        earliest_trace_p5 = pd.NA

    out_rows.append({
        'cmdb_id': cmdb,
        'kpi_name': kpi,
        'run_start_timestamp': run_start_dt.isoformat(),
        'run_end_timestamp': pd.to_datetime(run_end_str).tz_convert(tz).isoformat() if pd.notnull(run_end_str) else pd.NaT,
        'length_minutes': length_minutes,
        'severity_relative': severity,
        'earliest_trace_after_run_timestamp': earliest_trace_after_run_timestamp,
        'earliest_trace_name': earliest_trace_name,
        'earliest_trace_value': earliest_trace_value,
        'earliest_trace_p95': earliest_trace_p95,
        'earliest_trace_p5': earliest_trace_p5
    })

# Build output DataFrame, order by severity desc, limit 20 rows
out_df = pd.DataFrame(out_rows)
out_df = out_df.sort_values('severity_relative', ascending=False).head(20).reset_index(drop=True)

# Keep output compact: selected columns only
out_df[['cmdb_id','kpi_name','run_start_timestamp','run_end_timestamp','length_minutes','severity_relative',
        'earliest_trace_after_run_timestamp','earliest_trace_name','earliest_trace_value','earliest_trace_p95','earliest_trace_p5']]

```
Out[11]:
```
```python
# Compose a concise plain-English summary from the most recent correlation of runs -> traces (out_df)
try:
    out_df
except NameError:
    summary = "No run->trace correlation results available in the session."
else:
    if out_df.empty:
        summary = "No consecutive metric breach runs with matching downstream trace anomalies were found."
    else:
        # Build example lines from top 5 runs
        example_lines = []
        for i, r in out_df.head(5).iterrows():
            example_lines.append(
                f"{i+1}. {r['cmdb_id']} | KPI={r['kpi_name']} | run {r['run_start_timestamp']}→{r['run_end_timestamp']} ({int(r['length_minutes'])}m) | "
                f"severity={r['severity_relative']:.3g} | earliest trace after run: {r['earliest_trace_after_run_timestamp']} "
                f"({r['earliest_trace_name']} value={r['earliest_trace_value']}, p95={r['earliest_trace_p95']}, p5={r['earliest_trace_p5']})"
            )

        # Identify top candidates by severity (top 3)
        top_candidates = out_df.sort_values('severity_relative', ascending=False).head(3)['cmdb_id'].unique().tolist()

        summary_lines = [
            "Run→Trace correlation summary (incident window 2022-03-20 20:00–20:30 Asia/Shanghai):",
            "",
            "Top findings:",
            "- Highest-severity consecutive metric breaches (2+ minutes) with matching downstream trace anomalies were detected.",
            "- Priority components (top candidates): " + ", ".join(top_candidates) + ".",
            "",
            "Top examples:",
            *example_lines,
            "",
            "Interpretation & recommended next steps:",
            "- Immediate priority: Investigate currencyservice instances (currencyservice-0 and currencyservice-2).",
            "  * currencyservice-0 shows a network receive MB high run at ~20:10–20:11 with a matching trace to_currencyservice-0 at the same minute.",
            "  * currencyservice-2 shows CPU throttling high at ~20:05–20:06 with a matching trace to_currencyservice-2 at the same minute.",
            "- Secondary: productcatalogservice instances show repeated mesh destination/source anomalies (multiple 2-min runs) around 20:00–20:03 and 20:21–20:29; traces to those productcatalogservice targets occur within 0–5 minutes after metric run starts.",
            "- Because traces occur at or shortly after metric-run starts, these downstream-target observations strengthen suspicion that the listed components are involved in the fault path.",
            "",
            "Actionable actions:",
            "1) Start with currencyservice (CPU throttling and network receive) — inspect pod logs, CPU throttling metrics, and network I/O around the cited minutes.",
            "2) For productcatalogservice instances, inspect mesh/istio metrics, request traces, and host/container resource metrics around the listed run start times.",
            "3) Correlate with logs and traces for the exact minute-level timestamps shown above to confirm root cause and reproduce.",
        ]
        summary = "\n".join(summary_lines)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id                                           kpi_name        run_start_timestamp          run_end_timestamp  length_minutes  severity_relative earliest_trace_after_run_timestamp                             earliest_trace_name  earliest_trace_value  earliest_trace_p95  earliest_trace_p5
0          currencyservice-0  container.node-6.container_network_receive_MB....  2022-03-20T20:10:00+08:00  2022-03-20T20:11:00+08:00               2           2.264727          2022-03-20T20:10:00+08:00            trace.to_currencyservice-0.row_count              4.000000            3.000000           1.000000
1          currencyservice-2  container.node-6.container_cpu_cfs_throttled_s...  2022-03-20T20:05:00+08:00  2022-03-20T20:06:00+08:00               2           2.140148          2022-03-20T20:05:00+08:00        trace.to_currencyservice-2.duration_mean              0.000762            0.000152           0.000107
2    productcatalogservice-0  mesh.destination.checkoutservice.productcatalo...  2022-03-20T20:25:00+08:00  2022-03-20T20:26:00+08:00               2           0.222018          2022-03-20T20:25:00+08:00   trace.to_productcatalogservice-0.duration_p95              0.006490            0.006398           0.005300
3    productcatalogservice-2  mesh.destination.recommendationservice2.produc...  2022-03-20T20:00:00+08:00  2022-03-20T20:01:00+08:00               2           0.180952          2022-03-20T20:00:00+08:00  trace.to_productcatalogservice-2.duration_mean              0.004893            0.004836           0.004533
4          currencyservice-2  mesh.destination.frontend.currencyservice.isti...  2022-03-20T20:21:00+08:00  2022-03-20T20:22:00+08:00               2           0.074866          2022-03-20T20:21:00+08:00        trace.to_currencyservice-2.duration_mean              0.000144            0.000142           0.000115
5    productcatalogservice-2  mesh.source.productcatalogservice.jaeger-colle...  2022-03-20T20:02:00+08:00  2022-03-20T20:03:00+08:00               2           0.066667          2022-03-20T20:02:00+08:00  trace.to_productcatalogservice-2.duration_mean              0.004960            0.004822           0.004385
6    productcatalogservice-1  container.node-6.container_network_receive_MB....  2022-03-20T20:21:00+08:00  2022-03-20T20:22:00+08:00               2           0.059618          2022-03-20T20:21:00+08:00      trace.to_productcatalogservice-1.row_count             75.000000           74.550000          64.450000
7    productcatalogservice-1  container.node-6.container_network_transmit_pa...  2022-03-20T20:21:00+08:00  2022-03-20T20:22:00+08:00               2           0.059149          2022-03-20T20:21:00+08:00      trace.to_productcatalogservice-1.row_count             75.000000           74.550000          64.450000
8    productcatalogservice-0  mesh.source.productcatalogservice.jaeger-colle...  2022-03-20T20:03:00+08:00  2022-03-20T20:04:00+08:00               2           0.058824          2022-03-20T20:03:00+08:00  trace.to_productcatalogservice-0.duration_mean              0.004185            0.004854           0.004232
9    productcatalogservice-1  container.node-6.container_network_receive_pac...  2022-03-20T20:21:00+08:00  2022-03-20T20:22:00+08:00               2           0.054164          2022-03-20T20:21:00+08:00      trace.to_productcatalogservice-1.row_count             75.000000           74.550000          64.450000
10   productcatalogservice-0  mesh.destination.recommendationservice2.produc...  2022-03-20T20:29:00+08:00  2022-03-20T20:30:00+08:00               2           0.052239          2022-03-20T20:29:00+08:00  trace.to_productcatalogservice-0.duration_mean              0.004412            0.005568           0.004549
11   productcatalogservice-0  mesh.destination.recommendationservice2.produc...  2022-03-20T20:29:00+08:00  2022-03-20T20:30:00+08:00               2           0.052239          2022-03-20T20:29:00+08:00  trace.to_productcatalogservice-0.duration_mean              0.004412            0.005568           0.004549
12   productcatalogservice-0        container.node-6.container_cpu_user_seconds  2022-03-20T20:21:00+08:00  2022-03-20T20:22:00+08:00               2           0.048780          2022-03-20T20:22:00+08:00  trace.to_productcatalogservice-0.duration_mean              0.004245            0.005604           0.004537
13   productcatalogservice-2  mesh.destination.recommendationservice.product...  2022-03-20T20:23:00+08:00  2022-03-20T20:24:00+08:00               2           0.046154          2022-03-20T20:23:00+08:00  trace.to_productcatalogservice-2.duration_mean              0.005881            0.005818           0.004568
14   productcatalogservice-2  mesh.destination.recommendationservice.product...  2022-03-20T20:23:00+08:00  2022-03-20T20:24:00+08:00               2           0.046154          2022-03-20T20:23:00+08:00  trace.to_productcatalogservice-2.duration_mean              0.005881            0.005818           0.004568
15   productcatalogservice-2  mesh.destination.recommendationservice.product...  2022-03-20T20:23:00+08:00  2022-03-20T20:24:00+08:00               2           0.046154          2022-03-20T20:23:00+08:00  trace.to_productcatalogservice-2.duration_mean              0.005881            0.005818           0.004568
16  productcatalogservice2-0  mesh.source.productcatalogservice2.jaeger-coll...  2022-03-20T20:29:00+08:00  2022-03-20T20:30:00+08:00               2           0.039516          2022-03-20T20:29:00+08:00     trace.to_productcatalogservice2-0.row_count           1160.000000         1429.350000        1256.900000
17  productcatalogservice2-0  mesh.source.productcatalogservice2.basic-tidb....  2022-03-20T20:29:00+08:00  2022-03-20T20:30:00+08:00               2           0.038447          2022-03-20T20:29:00+08:00     trace.to_productcatalogservice2-0.row_count           1160.000000         1429.350000        1256.900000
18  productcatalogservice2-0  mesh.source.productcatalogservice2.basic-tidb....  2022-03-20T20:29:00+08:00  2022-03-20T20:30:00+08:00               2           0.038159          2022-03-20T20:29:00+08:00     trace.to_productcatalogservice2-0.row_count           1160.000000         1429.350000        1256.900000
19  productcatalogservice2-0  mesh.source.productcatalogservice2.basic-tidb....  2022-03-20T20:29:00+08:00  2022-03-20T20:30:00+08:00               2           0.038159          2022-03-20T20:29:00+08:00     trace.to_productcatalogservice2-0.row_count           1160.000000         1429.350000        1256.900000```
```